# AI Model 04 예측 모델 설계 및 학습


In [ ]:
# ============================================================
# AI Model 04 공통 경로 설정
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

ROOT_PATH = "/content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/"
DATA_PATH = os.path.join(ROOT_PATH, "data") + "/"
PROCESSED_PATH = os.path.join(ROOT_PATH, "preprocessed_data") + "/"
DATA_COLLECTION_PATH = os.path.join(ROOT_PATH, "data collection") + "/"

print(f"ROOT_PATH      = {ROOT_PATH}")
print(f"DATA_PATH      = {DATA_PATH}")
print(f"PROCESSED_PATH = {PROCESSED_PATH}")
print(f"DATA_COLLECTION_PATH = {DATA_COLLECTION_PATH}")

!pip -q install lightgbm duckdb pyarrow joblib


## 5단계: 예측 모델 생성

In [ ]:
# ============================================================
# 최종 격자 패널 진단 리포트
# 대상:
#   grid_station_daily_panel_500m_plus_facility_plus_geo_plus_official_price.parquet
#
# 목적:
#   예측 모델 재설계를 위해 필요한 스키마, 타깃, feature 후보,
#   결측, 날짜/격자 커버리지, 공시지가, 샘플, 기존 selector 기준 feature를 전부 출력
# ============================================================

from __future__ import annotations

import os
import sys
import json
import math
import textwrap
import subprocess
from pathlib import Path
from datetime import datetime

# ------------------------------------------------------------
# 0. 패키지 준비
# ------------------------------------------------------------
def ensure_package(import_name: str, pip_name: str | None = None):
    pip_name = pip_name or import_name
    try:
        __import__(import_name)
    except Exception:
        print(f"[INSTALL] {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

ensure_package("duckdb")
ensure_package("pyarrow")
ensure_package("pandas")
ensure_package("numpy")

import numpy as np
import pandas as pd
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 300)

# ------------------------------------------------------------
# 1. 사용자 경로 설정
# ------------------------------------------------------------
ROOT_PATH = Path(ROOT_PATH)
DATA_PATH = Path(DATA_PATH)
PROCESSED_PATH = Path(PROCESSED_PATH)
GRID_DIR = ROOT_PATH / "그리드"
DATA1_DIR = GRID_DIR / "data_1"

PANEL_PATH = (
    DATA1_DIR
    / "grid_station_daily_panel_500m_plus_facility_plus_geo_plus_official_price.parquet"
)

# 혹시 경로명이 다르면 여기만 직접 수정하면 됨.
# PANEL_PATH = Path("/직접/경로/grid_station_daily_panel_500m_plus_facility_plus_geo_plus_official_price.parquet")

if not PANEL_PATH.exists():
    # ROOT_PATH 아래에서 동일 파일명을 한 번 찾아본다.
    matches = list(ROOT_PATH.rglob("grid_station_daily_panel_500m_plus_facility_plus_geo_plus_official_price.parquet"))
    if len(matches) == 1:
        PANEL_PATH = matches[0]
    elif len(matches) > 1:
        print("[주의] 같은 이름의 parquet 파일이 여러 개 발견됨. 첫 번째를 사용합니다.")
        for m in matches:
            print(" -", m)
        PANEL_PATH = matches[0]
    else:
        raise FileNotFoundError(f"최종 parquet 파일을 찾지 못했습니다: {PANEL_PATH}")

REPORT_PATH = PANEL_PATH.parent / "panel_diagnostic_report_for_chatgpt.txt"

# ------------------------------------------------------------
# 2. 실행 옵션
# ------------------------------------------------------------
RUN_HEAVY_NUMERIC_PROFILE = True       # 전체 numeric 컬럼 분위수/통계
RUN_KEY_DUPLICATE_CHECK = True         # date-grid 중복 체크
RUN_TARGET_DAILY_TABLES = True         # 휘발유/경유 일자별 타깃 커버리지
RUN_CORRELATION_SAMPLE = True          # 샘플 기반 상관계수
SAMPLE_ROWS = 120_000                  # 상관계수용 최대 샘플 행 수
VALUE_COUNT_TOP_N = 50
NUMERIC_PROFILE_BATCH_SIZE = 12

# 예측 모델에서 사용하던 샘플링 비율 확인용
TRAIN_GRID_SAMPLE_PER_MILLE = 40

# ------------------------------------------------------------
# 3. 출력 helper
# ------------------------------------------------------------
report_lines: list[str] = []

def _now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def emit(title: str, body: str = ""):
    line = "\n" + "=" * 120 + f"\n[{title}]\n" + "=" * 120
    print(line)
    report_lines.append(line)
    if body:
        print(body)
        report_lines.append(str(body))

def df_text(df: pd.DataFrame, max_rows: int | None = None) -> str:
    if df is None:
        return "<None>"
    if len(df) == 0:
        return "<EMPTY DATAFRAME>"
    if max_rows is not None and len(df) > max_rows:
        shown = df.head(max_rows).copy()
        suffix = f"\n... ({len(df):,} rows total, showing first {max_rows:,})"
    else:
        shown = df.copy()
        suffix = ""
    with pd.option_context(
        "display.max_rows", 100000 if max_rows is None else max_rows + 5,
        "display.max_columns", 500,
        "display.width", 500,
        "display.max_colwidth", 200,
    ):
        return shown.to_string(index=False) + suffix

def emit_df(title: str, df: pd.DataFrame, max_rows: int | None = None):
    emit(title, df_text(df, max_rows=max_rows))

def qstr(s: str | Path) -> str:
    return "'" + str(s).replace("'", "''") + "'"

def qid(s: str) -> str:
    return '"' + str(s).replace('"', '""') + '"'

def chunked(xs, n):
    for i in range(0, len(xs), n):
        yield xs[i:i+n]

# ------------------------------------------------------------
# 4. 파일 / parquet metadata
# ------------------------------------------------------------
emit("REPORT_TEXT_START", f"created_at={_now()}")

file_stat = PANEL_PATH.stat()
file_info = pd.DataFrame([{
    "panel_path": str(PANEL_PATH),
    "exists": PANEL_PATH.exists(),
    "file_size_gb": round(file_stat.st_size / (1024 ** 3), 4),
    "file_size_mb": round(file_stat.st_size / (1024 ** 2), 2),
    "modified_at": datetime.fromtimestamp(file_stat.st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
}])
emit_df("FILE_INFO", file_info)

pf = pq.ParquetFile(PANEL_PATH)
meta = pf.metadata

parquet_info = pd.DataFrame([{
    "num_rows_metadata": meta.num_rows,
    "num_columns_metadata": meta.num_columns,
    "num_row_groups": meta.num_row_groups,
    "created_by": meta.created_by,
    "format_version": str(meta.format_version),
}])
emit_df("PARQUET_METADATA", parquet_info)

row_group_rows = []
for i in range(meta.num_row_groups):
    rg = meta.row_group(i)
    row_group_rows.append({
        "row_group": i,
        "num_rows": rg.num_rows,
        "total_byte_size_mb": round(rg.total_byte_size / (1024 ** 2), 2),
    })
row_group_df = pd.DataFrame(row_group_rows)
emit_df("PARQUET_ROW_GROUPS", row_group_df, max_rows=120)

arrow_schema_rows = []
for i, field in enumerate(pf.schema_arrow):
    arrow_schema_rows.append({
        "ord": i,
        "column_name": field.name,
        "arrow_type": str(field.type),
        "nullable": field.nullable,
    })
arrow_schema_df = pd.DataFrame(arrow_schema_rows)
emit_df("ARROW_SCHEMA_FULL", arrow_schema_df)

# ------------------------------------------------------------
# 5. DuckDB schema
# ------------------------------------------------------------
con = duckdb.connect()
con.execute("PRAGMA threads=4;")
con.execute(f"CREATE OR REPLACE VIEW panel AS SELECT * FROM read_parquet({qstr(PANEL_PATH)})")

schema = con.execute("DESCRIBE SELECT * FROM panel").df()
schema = schema.rename(columns={
    "column_name": "column_name",
    "column_type": "duckdb_type",
    "null": "duckdb_null",
    "key": "duckdb_key",
    "default": "duckdb_default",
    "extra": "duckdb_extra",
})
emit_df("DUCKDB_SCHEMA_FULL", schema)

cols = schema["column_name"].tolist()
type_map = dict(zip(schema["column_name"], schema["duckdb_type"]))

numeric_type_keywords = [
    "INTEGER", "BIGINT", "SMALLINT", "TINYINT", "HUGEINT",
    "DOUBLE", "FLOAT", "REAL", "DECIMAL", "UBIGINT", "UINTEGER", "USMALLINT", "UTINYINT"
]

def is_numeric_type(t: str) -> bool:
    tu = str(t).upper()
    return any(k in tu for k in numeric_type_keywords)

numeric_cols = [c for c in cols if is_numeric_type(type_map[c])]
non_numeric_cols = [c for c in cols if c not in numeric_cols]

col_type_summary = pd.DataFrame([{
    "n_total_cols": len(cols),
    "n_numeric_cols": len(numeric_cols),
    "n_non_numeric_cols": len(non_numeric_cols),
    "numeric_cols": ", ".join(numeric_cols),
    "non_numeric_cols": ", ".join(non_numeric_cols),
}])
emit_df("COLUMN_TYPE_SUMMARY", col_type_summary)

# ------------------------------------------------------------
# 6. 컬럼 그룹 자동 분류
# ------------------------------------------------------------
def classify_column(c: str) -> str:
    cl = c.lower()

    if c in ["date", "grid_id"]:
        return "01_key"
    if c in ["cell_x", "cell_y", "center_x", "center_y", "center_lon", "center_lat", "grid_col", "grid_row"]:
        return "02_grid_geometry"
    if c in ["gasoline_price_mean", "diesel_price_mean"]:
        return "03_target_grid_price"
    if c in ["gasoline_station_count", "diesel_station_count", "station_count_total"]:
        return "04_station_counts"
    if c.startswith("brand_count__"):
        return "05_brand_counts"
    if c.startswith("self_count__"):
        return "06_self_counts"
    if "price_mean__" in c:
        return "07_self_price_means_leakage_risk"
    if c.startswith("facility_") or c in ["storage_influence", "agency_influence", "factory_influence"]:
        return "08_facility_static"
    if c in ["is_sea", "is_island", "land_component_id"]:
        return "09_geo_flags"
    if c in ["official_land_price", "official_price_source_date"] or c.startswith("official_land_price__"):
        return "10_official_land_price"
    if "station_neighbor_influence" in c:
        return "11_station_neighbor_influence"
    if "price" in cl:
        return "12_other_price_related"
    return "99_other"

column_groups = pd.DataFrame({
    "column_name": cols,
    "duckdb_type": [type_map[c] for c in cols],
    "group": [classify_column(c) for c in cols],
})
column_groups = column_groups.sort_values(["group", "column_name"]).reset_index(drop=True)
emit_df("COLUMN_GROUPS", column_groups)

column_group_counts = (
    column_groups
    .groupby("group", as_index=False)
    .agg(n_cols=("column_name", "size"),
         columns=("column_name", lambda x: ", ".join(x)))
)
emit_df("COLUMN_GROUP_COUNTS", column_group_counts)

# ------------------------------------------------------------
# 7. 전체 row/date/grid summary
# ------------------------------------------------------------
select_exprs = ["COUNT(*) AS row_count"]

if "date" in cols:
    select_exprs += [
        "MIN(CAST(date AS DATE)) AS min_date",
        "MAX(CAST(date AS DATE)) AS max_date",
        "COUNT(DISTINCT CAST(date AS DATE)) AS unique_dates",
    ]

if "grid_id" in cols:
    select_exprs += ["COUNT(DISTINCT grid_id) AS unique_grid_id"]

if {"cell_x", "cell_y"}.issubset(cols):
    select_exprs += [
        "COUNT(*) FILTER (WHERE cell_x IS NULL OR cell_y IS NULL) AS null_cell_xy_rows",
        "MIN(cell_x) AS min_cell_x",
        "MAX(cell_x) AS max_cell_x",
        "MIN(cell_y) AS min_cell_y",
        "MAX(cell_y) AS max_cell_y",
    ]

if {"center_lon", "center_lat"}.issubset(cols):
    select_exprs += [
        "COUNT(*) FILTER (WHERE center_lon IS NULL OR center_lat IS NULL) AS null_center_lonlat_rows",
        "MIN(center_lon) AS min_center_lon",
        "MAX(center_lon) AS max_center_lon",
        "MIN(center_lat) AS min_center_lat",
        "MAX(center_lat) AS max_center_lat",
    ]

global_summary = con.execute(f"SELECT {', '.join(select_exprs)} FROM panel").df()
emit_df("GLOBAL_SUMMARY", global_summary)

if "date" in cols:
    date_df = con.execute("""
        SELECT DISTINCT CAST(date AS DATE) AS date
        FROM panel
        ORDER BY 1
    """).df()
    date_df["date"] = pd.to_datetime(date_df["date"])
    min_d = date_df["date"].min()
    max_d = date_df["date"].max()
    expected_dates = pd.date_range(min_d, max_d, freq="D")
    actual_dates = pd.DatetimeIndex(date_df["date"])
    missing_dates = expected_dates.difference(actual_dates)

    date_gap_summary = pd.DataFrame([{
        "min_date": min_d.date() if pd.notna(min_d) else None,
        "max_date": max_d.date() if pd.notna(max_d) else None,
        "expected_daily_dates": len(expected_dates),
        "actual_unique_dates": len(actual_dates),
        "missing_date_count": len(missing_dates),
        "first_50_missing_dates": ", ".join([str(x.date()) for x in missing_dates[:50]]),
    }])
    emit_df("DATE_GAP_SUMMARY", date_gap_summary)

# ------------------------------------------------------------
# 8. date-grid key 중복 체크
# ------------------------------------------------------------
if RUN_KEY_DUPLICATE_CHECK and {"date", "grid_id"}.issubset(cols):
    dup_key = con.execute("""
        WITH k AS (
            SELECT
                CAST(date AS DATE) AS date,
                grid_id,
                COUNT(*) AS cnt
            FROM panel
            GROUP BY 1, 2
        ),
        d AS (
            SELECT *
            FROM k
            WHERE cnt > 1
        )
        SELECT
            COUNT(*) AS duplicated_date_grid_keys,
            COALESCE(SUM(cnt - 1), 0) AS extra_duplicate_rows,
            COALESCE(MAX(cnt), 0) AS max_rows_per_date_grid_key
        FROM d
    """).df()
    emit_df("DATE_GRID_DUPLICATE_CHECK", dup_key)

    dup_examples = con.execute("""
        SELECT
            CAST(date AS DATE) AS date,
            grid_id,
            COUNT(*) AS cnt
        FROM panel
        GROUP BY 1, 2
        HAVING COUNT(*) > 1
        ORDER BY cnt DESC, date, grid_id
        LIMIT 30
    """).df()
    emit_df("DATE_GRID_DUPLICATE_EXAMPLES_TOP30", dup_examples)

# ------------------------------------------------------------
# 9. 일자별 row/grid/station coverage
# ------------------------------------------------------------
if "date" in cols:
    daily_exprs = [
        "CAST(date AS DATE) AS date",
        "COUNT(*) AS row_count",
    ]
    if "grid_id" in cols:
        daily_exprs.append("COUNT(DISTINCT grid_id) AS grid_count")
    if "station_count_total" in cols:
        daily_exprs.append("SUM(CAST(station_count_total AS DOUBLE)) AS station_count_total_sum")
        daily_exprs.append("AVG(CAST(station_count_total AS DOUBLE)) AS station_count_total_mean_per_row")
        daily_exprs.append("COUNT(*) FILTER (WHERE station_count_total > 0) AS rows_with_station")
    if "gasoline_station_count" in cols:
        daily_exprs.append("SUM(CAST(gasoline_station_count AS DOUBLE)) AS gasoline_station_count_sum")
        daily_exprs.append("COUNT(*) FILTER (WHERE gasoline_station_count > 0) AS rows_with_gasoline_station")
    if "diesel_station_count" in cols:
        daily_exprs.append("SUM(CAST(diesel_station_count AS DOUBLE)) AS diesel_station_count_sum")
        daily_exprs.append("COUNT(*) FILTER (WHERE diesel_station_count > 0) AS rows_with_diesel_station")

    daily_sql = f"""
        SELECT
            {", ".join(daily_exprs)}
        FROM panel
        GROUP BY 1
        ORDER BY 1
    """
    daily = con.execute(daily_sql).df()

    emit_df("DAILY_COVERAGE_DESCRIBE", daily.describe(include="all").T.reset_index().rename(columns={"index": "column"}))
    emit_df("DAILY_COVERAGE_HEAD20", daily.head(20))
    emit_df("DAILY_COVERAGE_TAIL20", daily.tail(20))

    if "row_count" in daily.columns:
        row_count_freq = (
            daily["row_count"]
            .value_counts(dropna=False)
            .rename_axis("row_count")
            .reset_index(name="n_dates")
            .sort_values(["n_dates", "row_count"], ascending=[False, True])
        )
        emit_df("DAILY_ROW_COUNT_FREQUENCY", row_count_freq)

        mode_row_count = int(row_count_freq.iloc[0]["row_count"])
        anomalous_days = daily.loc[daily["row_count"] != mode_row_count].copy()
        emit_df("DAILY_ROW_COUNT_ANOMALIES_NOT_MODE", anomalous_days, max_rows=100)

# ------------------------------------------------------------
# 10. grid별 coverage
# ------------------------------------------------------------
if {"date", "grid_id"}.issubset(cols):
    grid_coverage_summary = con.execute("""
        WITH gd AS (
            SELECT
                grid_id,
                COUNT(*) AS n_rows,
                COUNT(DISTINCT CAST(date AS DATE)) AS n_dates,
                MIN(CAST(date AS DATE)) AS min_date,
                MAX(CAST(date AS DATE)) AS max_date
            FROM panel
            GROUP BY 1
        )
        SELECT
            COUNT(*) AS n_grids,
            MIN(n_rows) AS min_rows_per_grid,
            APPROX_QUANTILE(n_rows, 0.01) AS p01_rows_per_grid,
            APPROX_QUANTILE(n_rows, 0.05) AS p05_rows_per_grid,
            APPROX_QUANTILE(n_rows, 0.50) AS median_rows_per_grid,
            APPROX_QUANTILE(n_rows, 0.95) AS p95_rows_per_grid,
            MAX(n_rows) AS max_rows_per_grid,
            MIN(n_dates) AS min_dates_per_grid,
            APPROX_QUANTILE(n_dates, 0.50) AS median_dates_per_grid,
            MAX(n_dates) AS max_dates_per_grid
        FROM gd
    """).df()
    emit_df("GRID_COVERAGE_SUMMARY", grid_coverage_summary)

    grid_coverage_examples = con.execute("""
        WITH gd AS (
            SELECT
                grid_id,
                COUNT(*) AS n_rows,
                COUNT(DISTINCT CAST(date AS DATE)) AS n_dates,
                MIN(CAST(date AS DATE)) AS min_date,
                MAX(CAST(date AS DATE)) AS max_date
            FROM panel
            GROUP BY 1
        )
        SELECT *
        FROM gd
        ORDER BY n_dates ASC, grid_id
        LIMIT 50
    """).df()
    emit_df("GRID_COVERAGE_SHORTEST_50", grid_coverage_examples)

# ------------------------------------------------------------
# 11. 결측 / distinct counts: 전체 컬럼
# ------------------------------------------------------------
null_parts = []
for c in cols:
    null_parts.append(f"""
        SELECT
            {qstr(c)} AS column_name,
            {qstr(str(type_map[c]))} AS duckdb_type,
            COUNT(*) AS total_rows,
            COUNT({qid(c)}) AS non_null_rows,
            COUNT(*) - COUNT({qid(c)}) AS null_rows,
            CAST(COUNT(*) - COUNT({qid(c)}) AS DOUBLE) / NULLIF(COUNT(*), 0) AS null_ratio,
            APPROX_COUNT_DISTINCT(CAST({qid(c)} AS VARCHAR)) AS approx_distinct_count
        FROM panel
    """)

null_profile = con.execute("\nUNION ALL\n".join(null_parts)).df()
null_profile = null_profile.sort_values(
    ["null_ratio", "column_name"],
    ascending=[False, True]
).reset_index(drop=True)

emit_df("NULL_PROFILE_ALL_COLUMNS", null_profile)

# ------------------------------------------------------------
# 12. numeric profile
# ------------------------------------------------------------
if RUN_HEAVY_NUMERIC_PROFILE and numeric_cols:
    numeric_profiles = []

    for batch_idx, batch_cols in enumerate(chunked(numeric_cols, NUMERIC_PROFILE_BATCH_SIZE), start=1):
        print(f"[numeric profile] batch {batch_idx} / {math.ceil(len(numeric_cols) / NUMERIC_PROFILE_BATCH_SIZE)}")

        parts = []
        for c in batch_cols:
            x = f"TRY_CAST({qid(c)} AS DOUBLE)"
            parts.append(f"""
                SELECT
                    {qstr(c)} AS column_name,
                    {qstr(str(type_map[c]))} AS duckdb_type,
                    COUNT(*) AS total_rows,
                    COUNT({qid(c)}) AS non_null_rows,
                    COUNT(*) - COUNT({qid(c)}) AS null_rows,
                    CAST(COUNT(*) - COUNT({qid(c)}) AS DOUBLE) / NULLIF(COUNT(*), 0) AS null_ratio,
                    APPROX_COUNT_DISTINCT({qid(c)}) AS approx_distinct_count,

                    SUM(CASE WHEN {x} = 0 THEN 1 ELSE 0 END) AS zero_count,
                    SUM(CASE WHEN {x} < 0 THEN 1 ELSE 0 END) AS negative_count,

                    AVG({x}) AS mean,
                    STDDEV_SAMP({x}) AS std,
                    MIN({x}) AS min,
                    APPROX_QUANTILE({x}, 0.01) AS p01,
                    APPROX_QUANTILE({x}, 0.05) AS p05,
                    APPROX_QUANTILE({x}, 0.10) AS p10,
                    APPROX_QUANTILE({x}, 0.25) AS p25,
                    APPROX_QUANTILE({x}, 0.50) AS p50,
                    APPROX_QUANTILE({x}, 0.75) AS p75,
                    APPROX_QUANTILE({x}, 0.90) AS p90,
                    APPROX_QUANTILE({x}, 0.95) AS p95,
                    APPROX_QUANTILE({x}, 0.99) AS p99,
                    MAX({x}) AS max
                FROM panel
            """)

        batch_df = con.execute("\nUNION ALL\n".join(parts)).df()
        numeric_profiles.append(batch_df)

    numeric_profile = pd.concat(numeric_profiles, ignore_index=True)
    numeric_profile = numeric_profile.sort_values("column_name").reset_index(drop=True)
    emit_df("NUMERIC_PROFILE_ALL_NUMERIC_COLUMNS", numeric_profile)

# ------------------------------------------------------------
# 13. 주요 categorical/value counts
# ------------------------------------------------------------
value_count_cols = [
    "official_price_source_date",
    "is_sea",
    "is_island",
    "land_component_id",
]

value_count_cols += [c for c in cols if c.startswith("brand_count__")]
value_count_cols += [c for c in cols if c.startswith("self_count__")]

for c in value_count_cols:
    if c not in cols:
        continue

    vc = con.execute(f"""
        SELECT
            CAST({qid(c)} AS VARCHAR) AS value,
            COUNT(*) AS row_count
        FROM panel
        GROUP BY 1
        ORDER BY row_count DESC, value
        LIMIT {VALUE_COUNT_TOP_N}
    """).df()
    emit_df(f"VALUE_COUNTS_TOP_{VALUE_COUNT_TOP_N}__{c}", vc)

# ------------------------------------------------------------
# 14. 공시지가 official_land_price 상세
# ------------------------------------------------------------
official_cols_exist = [c for c in ["official_land_price", "official_price_source_date"] if c in cols]
if official_cols_exist:
    official_summary_exprs = [
        "COUNT(*) AS total_rows",
    ]
    if "official_land_price" in cols:
        official_summary_exprs += [
            "COUNT(official_land_price) AS official_land_price_non_null_rows",
            "COUNT(*) - COUNT(official_land_price) AS official_land_price_null_rows",
            "AVG(CAST(official_land_price AS DOUBLE)) AS official_land_price_mean",
            "STDDEV_SAMP(CAST(official_land_price AS DOUBLE)) AS official_land_price_std",
            "MIN(CAST(official_land_price AS DOUBLE)) AS official_land_price_min",
            "APPROX_QUANTILE(CAST(official_land_price AS DOUBLE), 0.01) AS official_land_price_p01",
            "APPROX_QUANTILE(CAST(official_land_price AS DOUBLE), 0.05) AS official_land_price_p05",
            "APPROX_QUANTILE(CAST(official_land_price AS DOUBLE), 0.50) AS official_land_price_p50",
            "APPROX_QUANTILE(CAST(official_land_price AS DOUBLE), 0.95) AS official_land_price_p95",
            "APPROX_QUANTILE(CAST(official_land_price AS DOUBLE), 0.99) AS official_land_price_p99",
            "MAX(CAST(official_land_price AS DOUBLE)) AS official_land_price_max",
        ]
    if "official_price_source_date" in cols:
        official_summary_exprs += [
            "COUNT(official_price_source_date) AS official_price_source_date_non_null_rows",
            "COUNT(DISTINCT official_price_source_date) AS official_price_source_date_distinct",
            "MIN(official_price_source_date) AS min_official_price_source_date",
            "MAX(official_price_source_date) AS max_official_price_source_date",
        ]

    official_summary = con.execute(f"""
        SELECT {", ".join(official_summary_exprs)}
        FROM panel
    """).df()
    emit_df("OFFICIAL_PRICE_SUMMARY", official_summary)

    if "date" in cols and "official_land_price" in cols:
        official_daily = con.execute("""
            SELECT
                CAST(date AS DATE) AS date,
                COUNT(*) AS row_count,
                COUNT(official_land_price) AS official_non_null_rows,
                COUNT(*) - COUNT(official_land_price) AS official_null_rows,
                CAST(COUNT(*) - COUNT(official_land_price) AS DOUBLE) / NULLIF(COUNT(*), 0) AS official_null_ratio,
                AVG(CAST(official_land_price AS DOUBLE)) AS official_mean,
                APPROX_QUANTILE(CAST(official_land_price AS DOUBLE), 0.50) AS official_median
            FROM panel
            GROUP BY 1
            ORDER BY 1
        """).df()
        emit_df("OFFICIAL_PRICE_DAILY_HEAD20", official_daily.head(20))
        emit_df("OFFICIAL_PRICE_DAILY_TAIL20", official_daily.tail(20))
        emit_df("OFFICIAL_PRICE_DAILY_NULL_RATIO_DESCRIBE", official_daily[["official_null_ratio", "official_mean", "official_median"]].describe().T.reset_index().rename(columns={"index": "metric"}))

        official_null_days = official_daily[official_daily["official_null_ratio"] > 0].copy()
        emit_df("OFFICIAL_PRICE_DAILY_WITH_NULLS_FIRST100", official_null_days.head(100))

# ------------------------------------------------------------
# 15. 예측모델용 target / weight 후보 상세
# ------------------------------------------------------------
FUEL_CONFIG_FOR_INSPECTION = {
    "gasoline": {
        "label": "휘발유",
        "grid_price_col": "gasoline_price_mean",
        "grid_count_col": "gasoline_station_count",
    },
    "diesel": {
        "label": "경유",
        "grid_price_col": "diesel_price_mean",
        "grid_count_col": "diesel_station_count",
    },
}

target_summary_rows = []

for fuel_key, cfg in FUEL_CONFIG_FOR_INSPECTION.items():
    target_col = cfg["grid_price_col"]
    count_col = cfg["grid_count_col"]

    if target_col not in cols or count_col not in cols:
        target_summary_rows.append({
            "fuel_key": fuel_key,
            "label": cfg["label"],
            "status": "missing_target_or_count_col",
            "target_col": target_col,
            "count_col": count_col,
        })
        continue

    target_summary = con.execute(f"""
        SELECT
            {qstr(fuel_key)} AS fuel_key,
            {qstr(cfg["label"])} AS label,
            {qstr(target_col)} AS target_col,
            {qstr(count_col)} AS count_col,

            COUNT(*) AS total_rows,
            COUNT({qid(target_col)}) AS target_non_null_rows,
            COUNT(*) - COUNT({qid(target_col)}) AS target_null_rows,
            COUNT({qid(count_col)}) AS count_non_null_rows,

            COUNT(*) FILTER (WHERE {qid(count_col)} > 0) AS count_positive_rows,
            SUM(CAST({qid(count_col)} AS DOUBLE)) AS station_weight_total,

            MIN(CASE WHEN {qid(target_col)} IS NOT NULL THEN CAST(date AS DATE) ELSE NULL END) AS target_min_date,
            MAX(CASE WHEN {qid(target_col)} IS NOT NULL THEN CAST(date AS DATE) ELSE NULL END) AS target_max_date,
            COUNT(DISTINCT CASE WHEN {qid(target_col)} IS NOT NULL THEN CAST(date AS DATE) ELSE NULL END) AS target_unique_dates,
            COUNT(DISTINCT CASE WHEN {qid(target_col)} IS NOT NULL THEN grid_id ELSE NULL END) AS target_unique_grids,

            AVG(CAST({qid(target_col)} AS DOUBLE)) AS target_mean,
            STDDEV_SAMP(CAST({qid(target_col)} AS DOUBLE)) AS target_std,
            MIN(CAST({qid(target_col)} AS DOUBLE)) AS target_min,
            APPROX_QUANTILE(CAST({qid(target_col)} AS DOUBLE), 0.01) AS target_p01,
            APPROX_QUANTILE(CAST({qid(target_col)} AS DOUBLE), 0.05) AS target_p05,
            APPROX_QUANTILE(CAST({qid(target_col)} AS DOUBLE), 0.50) AS target_p50,
            APPROX_QUANTILE(CAST({qid(target_col)} AS DOUBLE), 0.95) AS target_p95,
            APPROX_QUANTILE(CAST({qid(target_col)} AS DOUBLE), 0.99) AS target_p99,
            MAX(CAST({qid(target_col)} AS DOUBLE)) AS target_max,

            AVG(CAST({qid(count_col)} AS DOUBLE)) AS count_mean,
            APPROX_QUANTILE(CAST({qid(count_col)} AS DOUBLE), 0.50) AS count_p50,
            MAX(CAST({qid(count_col)} AS DOUBLE)) AS count_max
        FROM panel
    """).df()

    target_summary_rows.append(target_summary.iloc[0].to_dict())

target_summary_df = pd.DataFrame(target_summary_rows)
emit_df("TARGET_SUMMARY_GASOLINE_DIESEL", target_summary_df)

if RUN_TARGET_DAILY_TABLES and "date" in cols:
    for fuel_key, cfg in FUEL_CONFIG_FOR_INSPECTION.items():
        target_col = cfg["grid_price_col"]
        count_col = cfg["grid_count_col"]

        if target_col not in cols or count_col not in cols:
            continue

        daily_target = con.execute(f"""
            SELECT
                CAST(date AS DATE) AS date,
                COUNT(*) AS row_count,
                COUNT({qid(target_col)}) AS target_non_null_rows,
                COUNT(*) - COUNT({qid(target_col)}) AS target_null_rows,
                COUNT(*) FILTER (WHERE {qid(count_col)} > 0) AS rows_with_positive_station_count,
                SUM(CAST({qid(count_col)} AS DOUBLE)) AS station_weight_sum,
                AVG(CAST({qid(target_col)} AS DOUBLE)) AS target_unweighted_mean,
                SUM(CAST({qid(target_col)} AS DOUBLE) * CAST({qid(count_col)} AS DOUBLE))
                    / NULLIF(SUM(CAST({qid(count_col)} AS DOUBLE)), 0) AS target_weighted_mean
            FROM panel
            GROUP BY 1
            ORDER BY 1
        """).df()

        emit_df(f"DAILY_TARGET_COVERAGE_DESCRIBE__{fuel_key}", daily_target.describe().T.reset_index().rename(columns={"index": "metric"}))
        emit_df(f"DAILY_TARGET_COVERAGE_HEAD20__{fuel_key}", daily_target.head(20))
        emit_df(f"DAILY_TARGET_COVERAGE_TAIL20__{fuel_key}", daily_target.tail(20))

        low_coverage = daily_target.sort_values(
            ["target_non_null_rows", "date"],
            ascending=[True, True]
        ).head(80)
        emit_df(f"DAILY_TARGET_LOWEST_COVERAGE_80__{fuel_key}", low_coverage)

# ------------------------------------------------------------
# 16. 기존 5단계 selector 기준 feature 후보 / 제외 컬럼
# ------------------------------------------------------------
leakage_keywords = [
    "price_mean",
    "actual",
    "pred",
    "fair",
    "적정",
    "하한",
    "상한",
    "deviation",
    "inside",
    "above",
    "below",
    "over",
    "under",
    "residual",
    "target",
    "spread",
    "judge",
    "band",
]

selector_rows = []
selector_summary_rows = []

for fuel_key, cfg in FUEL_CONFIG_FOR_INSPECTION.items():
    always_exclude = {
        "date",
        "grid_id",
        cfg["grid_price_col"],
    }

    selected = []
    excluded = []

    for col in numeric_cols:
        col_lower = col.lower()
        reason = None

        if col in always_exclude:
            reason = "always_exclude"
        elif any(k.lower() in col_lower for k in leakage_keywords):
            reason = "leakage_keyword"

        if reason is None:
            selected.append(col)
            selector_rows.append({
                "fuel_key": fuel_key,
                "label": cfg["label"],
                "column_name": col,
                "duckdb_type": type_map[col],
                "selector_result": "selected",
                "reason": "",
                "column_group": classify_column(col),
            })
        else:
            excluded.append((col, reason))
            selector_rows.append({
                "fuel_key": fuel_key,
                "label": cfg["label"],
                "column_name": col,
                "duckdb_type": type_map[col],
                "selector_result": "excluded",
                "reason": reason,
                "column_group": classify_column(col),
            })

    selector_summary_rows.append({
        "fuel_key": fuel_key,
        "label": cfg["label"],
        "n_numeric_cols": len(numeric_cols),
        "n_selected_base_features": len(selected),
        "n_excluded_numeric_cols": len(excluded),
        "selected_base_features": ", ".join(selected),
        "excluded_numeric_cols_with_reason": ", ".join([f"{c}:{r}" for c, r in excluded]),
    })

selector_summary_df = pd.DataFrame(selector_summary_rows)
selector_df = pd.DataFrame(selector_rows)

emit_df("MODEL_SELECTOR_SUMMARY_MIRRORING_EXISTING_CODE", selector_summary_df)
emit_df("MODEL_SELECTOR_DETAIL_SELECTED_AND_EXCLUDED", selector_df.sort_values(["fuel_key", "selector_result", "column_group", "column_name"]))

# ------------------------------------------------------------
# 17. 훈련 샘플링 hash 기준 grid 수 확인
# ------------------------------------------------------------
if "grid_id" in cols:
    sample_grid_summary = con.execute(f"""
        SELECT
            COUNT(DISTINCT grid_id) AS total_unique_grids,
            COUNT(DISTINCT CASE WHEN (hash(grid_id) % 1000) < {int(TRAIN_GRID_SAMPLE_PER_MILLE)} THEN grid_id ELSE NULL END) AS selected_unique_grids_at_{TRAIN_GRID_SAMPLE_PER_MILLE}_per_mille,
            CAST(COUNT(DISTINCT CASE WHEN (hash(grid_id) % 1000) < {int(TRAIN_GRID_SAMPLE_PER_MILLE)} THEN grid_id ELSE NULL END) AS DOUBLE)
                / NULLIF(COUNT(DISTINCT grid_id), 0) AS selected_grid_ratio
        FROM panel
    """).df()
    emit_df("HASH_GRID_SAMPLE_SUMMARY_FOR_TRAINING", sample_grid_summary)

    for fuel_key, cfg in FUEL_CONFIG_FOR_INSPECTION.items():
        target_col = cfg["grid_price_col"]
        count_col = cfg["grid_count_col"]
        if target_col not in cols or count_col not in cols:
            continue

        sample_target = con.execute(f"""
            SELECT
                {qstr(fuel_key)} AS fuel_key,
                COUNT(*) AS sampled_candidate_rows,
                COUNT(DISTINCT grid_id) AS sampled_grids,
                COUNT(DISTINCT CAST(date AS DATE)) AS sampled_dates,
                MIN(CAST(date AS DATE)) AS min_date,
                MAX(CAST(date AS DATE)) AS max_date
            FROM panel
            WHERE
                {qid(target_col)} IS NOT NULL
                AND {qid(count_col)} IS NOT NULL
                AND CAST({qid(count_col)} AS DOUBLE) > 0
                AND (hash(grid_id) % 1000) < {int(TRAIN_GRID_SAMPLE_PER_MILLE)}
        """).df()
        emit_df(f"HASH_SAMPLED_TARGET_ROWS__{fuel_key}", sample_target)

# ------------------------------------------------------------
# 18. 주요 feature sanity
# ------------------------------------------------------------
sanity_cols = [
    "station_count_total",
    "gasoline_station_count",
    "diesel_station_count",
    "facility_count_total",
    "facility_storage_count",
    "facility_factory_count",
    "facility_agency_count",
    "storage_influence",
    "agency_influence",
    "factory_influence",
    "is_sea",
    "is_island",
    "official_land_price",
]
sanity_cols = [c for c in sanity_cols if c in cols]

if sanity_cols:
    sanity_parts = []
    for c in sanity_cols:
        x = f"TRY_CAST({qid(c)} AS DOUBLE)"
        sanity_parts.append(f"""
            SELECT
                {qstr(c)} AS column_name,
                COUNT({qid(c)}) AS non_null,
                AVG({x}) AS mean,
                MIN({x}) AS min,
                APPROX_QUANTILE({x}, 0.50) AS p50,
                APPROX_QUANTILE({x}, 0.95) AS p95,
                MAX({x}) AS max,
                SUM(CASE WHEN {x} > 0 THEN 1 ELSE 0 END) AS positive_rows
            FROM panel
        """)
    sanity_df = con.execute("\nUNION ALL\n".join(sanity_parts)).df()
    emit_df("KEY_FEATURE_SANITY", sanity_df)

# ------------------------------------------------------------
# 19. 샘플 rows
# ------------------------------------------------------------
base_sample_cols = [
    "date", "grid_id", "cell_x", "cell_y", "center_lon", "center_lat",
    "station_count_total",
    "gasoline_station_count", "diesel_station_count",
    "gasoline_price_mean", "diesel_price_mean",
    "facility_count_total", "facility_storage_count", "facility_factory_count", "facility_agency_count",
    "storage_influence", "agency_influence", "factory_influence",
    "is_sea", "is_island", "land_component_id",
    "official_land_price", "official_price_source_date",
]
base_sample_cols = [c for c in base_sample_cols if c in cols]
select_sample_cols = ", ".join(qid(c) for c in base_sample_cols)

sample_head = con.execute(f"""
    SELECT {select_sample_cols}
    FROM panel
    ORDER BY CAST(date AS DATE), grid_id
    LIMIT 30
""").df()
emit_df("SAMPLE_HEAD_30_ORDERED", sample_head)

if "date" in cols:
    latest_date_df = con.execute("SELECT MAX(CAST(date AS DATE)) AS max_date FROM panel").df()
    latest_date = latest_date_df["max_date"].iloc[0]

    latest_sample = con.execute(f"""
        SELECT {select_sample_cols}
        FROM panel
        WHERE CAST(date AS DATE) = CAST({qstr(str(latest_date))} AS DATE)
        ORDER BY station_count_total DESC NULLS LAST, grid_id
        LIMIT 50
    """).df()
    emit_df("SAMPLE_LATEST_DATE_TOP_STATION_ROWS_50", latest_sample)

if "gasoline_price_mean" in cols:
    gas_sample = con.execute(f"""
        SELECT {select_sample_cols}
        FROM panel
        WHERE gasoline_price_mean IS NOT NULL
        ORDER BY CAST(date AS DATE) DESC, gasoline_station_count DESC NULLS LAST
        LIMIT 50
    """).df()
    emit_df("SAMPLE_GASOLINE_TARGET_NON_NULL_50", gas_sample)

if "diesel_price_mean" in cols:
    diesel_sample = con.execute(f"""
        SELECT {select_sample_cols}
        FROM panel
        WHERE diesel_price_mean IS NOT NULL
        ORDER BY CAST(date AS DATE) DESC, diesel_station_count DESC NULLS LAST
        LIMIT 50
    """).df()
    emit_df("SAMPLE_DIESEL_TARGET_NON_NULL_50", diesel_sample)

if "is_island" in cols:
    island_sample = con.execute(f"""
        SELECT {select_sample_cols}
        FROM panel
        WHERE is_island = 1
        ORDER BY CAST(date AS DATE) DESC, station_count_total DESC NULLS LAST, grid_id
        LIMIT 50
    """).df()
    emit_df("SAMPLE_ISLAND_ROWS_50", island_sample)

if "official_land_price" in cols:
    official_missing_sample = con.execute(f"""
        SELECT {select_sample_cols}
        FROM panel
        WHERE official_land_price IS NULL
        ORDER BY CAST(date AS DATE), grid_id
        LIMIT 50
    """).df()
    emit_df("SAMPLE_OFFICIAL_LAND_PRICE_NULL_ROWS_50", official_missing_sample)

# ------------------------------------------------------------
# 20. 샘플 기반 상관계수
# ------------------------------------------------------------
if RUN_CORRELATION_SAMPLE:
    selected_features_union = sorted(set(
        selector_df.loc[selector_df["selector_result"] == "selected", "column_name"].tolist()
    ))

    # 너무 많은 경우 상위 120개까지만. 현재는 보통 60개 안팎일 가능성이 큼.
    corr_feature_cols = [c for c in selected_features_union if c in cols and c in numeric_cols]
    corr_feature_cols = corr_feature_cols[:120]

    for fuel_key, cfg in FUEL_CONFIG_FOR_INSPECTION.items():
        target_col = cfg["grid_price_col"]
        if target_col not in cols:
            continue

        corr_cols = [target_col] + corr_feature_cols
        corr_cols = list(dict.fromkeys([c for c in corr_cols if c in cols]))

        if len(corr_cols) <= 1:
            continue

        corr_select = ", ".join([f"TRY_CAST({qid(c)} AS DOUBLE) AS {qid(c)}" for c in corr_cols])

        # 안정적이고 빠른 hash-grid sample. target non-null만.
        corr_sql = f"""
            SELECT {corr_select}
            FROM panel
            WHERE
                {qid(target_col)} IS NOT NULL
                AND (hash(grid_id) % 1000) < 80
            LIMIT {int(SAMPLE_ROWS)}
        """

        corr_sample = con.execute(corr_sql).df()

        if len(corr_sample) < 100:
            emit(f"CORRELATION_SAMPLE__{fuel_key}", f"sample rows too small: {len(corr_sample)}")
            continue

        corr = corr_sample.corr(numeric_only=True)[target_col].drop(labels=[target_col], errors="ignore")
        corr_df = (
            corr
            .dropna()
            .to_frame("corr_with_target_price")
            .reset_index()
            .rename(columns={"index": "feature"})
        )
        corr_df["abs_corr"] = corr_df["corr_with_target_price"].abs()
        corr_df = corr_df.sort_values("abs_corr", ascending=False).head(60).reset_index(drop=True)

        corr_meta = pd.DataFrame([{
            "fuel_key": fuel_key,
            "target_col": target_col,
            "sample_rows": len(corr_sample),
            "n_features_evaluated": len(corr_cols) - 1,
            "sample_rule": "(hash(grid_id) % 1000) < 80 and target non-null",
        }])

        emit_df(f"CORRELATION_SAMPLE_META__{fuel_key}", corr_meta)
        emit_df(f"TOP_ABS_CORRELATIONS_WITH_TARGET_PRICE__{fuel_key}", corr_df)

# ------------------------------------------------------------
# 21. 기존 selector의 위험 후보 확인
# ------------------------------------------------------------
# 기존 selector는 price_mean만 제외하므로 official_land_price는 선택될 가능성이 큼.
# 모델 재설계 시 이 컬럼을 쓸지/변환할지 판단하려고 명시 출력한다.
potentially_sensitive = []
for c in numeric_cols:
    cl = c.lower()
    if (
        "official" in cl
        or c in ["cell_x", "cell_y", "center_x", "center_y", "center_lon", "center_lat"]
        or "count" in cl
        or "influence" in cl
        or c in ["is_island", "is_sea", "land_component_id"]
    ):
        potentially_sensitive.append(c)

sensitive_df = pd.DataFrame({
    "column_name": potentially_sensitive,
    "duckdb_type": [type_map[c] for c in potentially_sensitive],
    "group": [classify_column(c) for c in potentially_sensitive],
    "note": [
        (
            "공간좌표/토지가격/시설/개수/섬 여부 계열. "
            "모델 feature로 쓸 수는 있으나 leakage·외삽·해석 가능성 관점에서 검토 필요"
        )
        for _ in potentially_sensitive
    ],
})
emit_df("POTENTIALLY_SENSITIVE_FEATURES_TO_REVIEW", sensitive_df)

# ------------------------------------------------------------
# 22. 추천 경로 출력
# ------------------------------------------------------------
path_recommendation = pd.DataFrame([{
    "recommended_GRID_PANEL_PATH_for_new_stage5": str(PANEL_PATH),
    "note": "새 예측 모델 단계에서는 기존 plus_geo 파일이 아니라 plus_geo_plus_official_price 최종 파일을 기준으로 시작",
}])
emit_df("RECOMMENDED_PATH_FOR_NEW_MODEL_STAGE", path_recommendation)

# ------------------------------------------------------------
# 23. 종료
# ------------------------------------------------------------
con.close()

emit("REPORT_TEXT_END", f"report_saved_to={REPORT_PATH}")

REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")

print("\n" + "#" * 120)
print(f"[저장 완료] {REPORT_PATH}")
print("위 출력에서 REPORT_TEXT_START ~ REPORT_TEXT_END 전체를 복사해서 보내주면 됩니다.")
print("#" * 120)

#### 환경설정

In [ ]:
# ============================================================
# 예측 모델 생성 - 셀 1. 설치 / import
# ============================================================

!pip -q install lightgbm duckdb pyarrow joblib

from __future__ import annotations

import os
import gc
import json
import math
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import duckdb
import lightgbm as lgb
import joblib

import pyarrow as pa
import pyarrow.parquet as pq

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("[OK] import 완료")

In [ ]:
# ============================================================
# 예측 모델 생성 - 셀 2. 경로 / 기본 설정 (v2 + 정책모듈)
# ============================================================

ROOT_PATH = "/content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/"
ROOT_PATH = Path(ROOT_PATH)

PROCESSED_PATH = ROOT_PATH / "preprocessed_data"
GRID_DIR = ROOT_PATH / "그리드"
DATA1_DIR = GRID_DIR / "data_1"

# 현재 노트북에서 생성한 최종 그리드 패널
GRID_PANEL_PATH = DATA1_DIR / "grid_station_daily_panel_500m_plus_facility_plus_geo.parquet"

# 전국 평균 실제 가격이 들어있는 통합 데이터
# (v2에서도 파일명/위치가 같다면 그대로 사용)
INTEGRATED_DAILY_PATH = PROCESSED_PATH / "분석용_일별_통합데이터.csv"

# ------------------------------------------------------------
# v2 결과 경로
# ------------------------------------------------------------
STEP2_DIR = ROOT_PATH / "적정가격대선정_v2"
POLICY_DIR = ROOT_PATH / "정책적용_v2"

# 이번 단계 저장 폴더
GRID_FAIR_OUTPUT_DIR = GRID_DIR / "적정 가격 예측_v2"
GRID_FAIR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 최종 test / 시각화 기간
# ------------------------------------------------------------
FINAL_TEST_START = "2025-01-01"
FINAL_TEST_END = "2026-12-31"

# ------------------------------------------------------------
# 학습 샘플링
# ------------------------------------------------------------
TRAIN_GRID_SAMPLE_PER_MILLE = 40

# LightGBM tuning fold
N_VALID_FOLDS = 4
VALID_DAYS = 180
GAP_DAYS = 7

# local interval coverage
LOCAL_INTERVAL_COVERAGE = 0.60

# feature selection
DO_FEATURE_SELECTION = True
MAX_SELECTED_FEATURES = 60
MIN_PERMUTATION_DELTA_MAE = 0.005

# 예측 결과에 selected feature도 같이 저장할지
SAVE_SELECTED_FEATURES_IN_OUTPUT = True

# 2025~2026 지도 시각화 날짜
MAP_PLOT_DATES = [
    "2025-01-15",
    "2025-07-15",
    "2026-03-20",
]

# ------------------------------------------------------------
# 유종 설정
# ------------------------------------------------------------
FUEL_CONFIG = {
    "gasoline": {
        "label": "휘발유",
        "grid_price_col": "gasoline_price_mean",
        "grid_count_col": "gasoline_station_count",
        "national_actual_col": "보통휘발유_평균",
        # 3단계 일별 정책적용 결과
        "policy_daily_path": POLICY_DIR / "휘발유" / "일별_정책적용_데이터_휘발유.csv",
        # 2단계 raw 결과 (fallback 용도, 기본은 안 씀)
        "step2_path": STEP2_DIR / "gasoline_production_predictions_full_calendar.csv",
        # 최고가격제 점검 파일
        "policy_cap_check_path": POLICY_DIR / "정유사_최고가격제_점검" / "정유사_최고가격제_점검_휘발유.csv",
    },
    "diesel": {
        "label": "경유",
        "grid_price_col": "diesel_price_mean",
        "grid_count_col": "diesel_station_count",
        "national_actual_col": "자동차용경유_평균",
        "policy_daily_path": POLICY_DIR / "경유" / "일별_정책적용_데이터_경유.csv",
        "step2_path": STEP2_DIR / "diesel_production_predictions_full_calendar.csv",
        "policy_cap_check_path": POLICY_DIR / "정유사_최고가격제_점검" / "정유사_최고가격제_점검_경유.csv",
    },
}

# ------------------------------------------------------------
# 전국 적정가격 입력 기준
# 반드시 3단계(정책적용_v2) 파일을 우선 사용
#   policy_applied : 3단계 정책적용 적정가격/범위 사용
#   nonpolicy      : 3단계 미정책 적정가격/범위 사용
# ------------------------------------------------------------
NATIONAL_FAIR_MODE = "policy_applied"

# 3단계 파일이 없으면 에러
REQUIRE_STAGE3_POLICY_FILE = True

# 혹시 나중에 디버깅용으로만 쓰고 싶으면 True로
ALLOW_STEP2_FALLBACK_IF_STAGE3_MISSING = False

# ------------------------------------------------------------
# 정책 feature 처리 방식
# carry_only : 정책 변수는 merge/저장/후처리에만 사용 (권장)
# model_input: 정책 변수도 모델 입력에 추가
#
# 주의:
# 정책기간을 전부 학습에서 배제하면 정책변수는 학습에서 거의 0/False라
# 모델이 정책 효과를 '학습'하지 못함.
# 그래서 기본값은 carry_only로 둠.
# ------------------------------------------------------------
POLICY_FEATURE_MODE = "carry_only"   # "carry_only" or "model_input"

# ------------------------------------------------------------
# 최고가격제 후처리
# rolling_margin_from_refinery:
#   정유사 최고가격제 점검 CSV에서
#   (주간 소매평균 - 정유사 실제 세후추정가격)의 최근 rolling median을 downstream margin proxy로 두고
#   retail_cap_approx = refinery_cap + margin_proxy
#   로 대충 retail cap 근사 후 hard clip
#
# 이 함수는 나중에 쉽게 교체 가능하게 별도 함수로 분리해둠.
# ------------------------------------------------------------
APPLY_PRICE_CAP_POSTPROCESS = True
PRICE_CAP_POSTPROCESS_MODE = "rolling_margin_from_refinery"
PRICE_CAP_MARGIN_LOOKBACK_WEEKS = 12
PRICE_CAP_MARGIN_MIN_WEEKS = 4

# ------------------------------------------------------------
# 정책 numeric/text columns
# ------------------------------------------------------------
POLICY_NUMERIC_FEATURES = [
    "fuel_tax_cut_pct",
    "fuel_tax_policy_on",
    "policy_shift_won_l",
    "price_cap_on",
    "refinery_price_cap_won_l",
    "price_cap_value_reflected",
    "policy_active_any",
    "retail_margin_proxy_won_l",
    "retail_price_cap_approx_won_l",
]

POLICY_TEXT_COLUMNS = [
    "fuel_tax_policy_name",
    "price_cap_policy_name",
    "policy_label",
]

# ------------------------------------------------------------
# 정책 적용 기간: 학습에서 항상 제외
# (네가 준 EXCLUDE_RANGES_STAGE1 그대로)
# ------------------------------------------------------------
POLICY_EXCLUDE_RANGES = [
    {"start": "2008-03-10", "end": "2008-12-31", "label": "fuel_tax_cut_2008"},
    {"start": "2011-04-07", "end": "2011-07-06", "label": "유류세 100원 인하"},
    {"start": "2018-11-06", "end": "2019-05-06", "label": "fuel_tax_cut_2018_15pct"},
    {"start": "2019-05-07", "end": "2019-08-31", "label": "fuel_tax_cut_2019_7pct"},
    {"start": "2021-11-12", "end": "2022-04-30", "label": "fuel_tax_cut_2021_20pct"},
    {"start": "2022-05-01", "end": "2022-06-30", "label": "fuel_tax_cut_2022_30pct"},
    {"start": "2022-07-01", "end": "2022-12-31", "label": "fuel_tax_cut_2022_37pct"},
    {"start": "2023-01-01", "end": "2024-06-30", "label": "fuel_tax_cut_maintained_2023_2024"},
    {"start": "2024-07-01", "end": "2024-10-31", "label": "fuel_tax_cut_2024_partial"},
    {"start": "2024-11-01", "end": "2025-04-30", "label": "fuel_tax_cut_2024_2025_partial"},
    {"start": "2025-05-01", "end": "2025-10-31", "label": "fuel_tax_cut_2025_readjusted"},
    {"start": "2025-11-01", "end": "2026-04-30", "label": "fuel_tax_cut_2025_2026_partial"},
    {"start": "2026-03-13", "end": "2026-03-26", "label": "price_cap_2026_round1"},
]

assert GRID_PANEL_PATH.exists(), f"그리드 패널 없음: {GRID_PANEL_PATH}"
assert INTEGRATED_DAILY_PATH.exists(), f"통합 일별 데이터 없음: {INTEGRATED_DAILY_PATH}"

for fuel_key, cfg in FUEL_CONFIG.items():
    if REQUIRE_STAGE3_POLICY_FILE:
        assert cfg["policy_daily_path"].exists(), f"[{fuel_key}] 3단계 정책 파일 없음: {cfg['policy_daily_path']}"
    if APPLY_PRICE_CAP_POSTPROCESS and (not cfg["policy_cap_check_path"].exists()):
        print(f"[WARN] [{fuel_key}] 최고가격제 점검 파일 없음: {cfg['policy_cap_check_path']} -> cap postprocess는 자동 skip")

print("[OK] 설정 완료")
print(f"[입력] GRID_PANEL_PATH = {GRID_PANEL_PATH}")
print(f"[입력] INTEGRATED_DAILY_PATH = {INTEGRATED_DAILY_PATH}")
print(f"[입력] STEP2_DIR = {STEP2_DIR}")
print(f"[입력] POLICY_DIR = {POLICY_DIR}")
print(f"[출력] GRID_FAIR_OUTPUT_DIR = {GRID_FAIR_OUTPUT_DIR}")
print(f"[설정] NATIONAL_FAIR_MODE = {NATIONAL_FAIR_MODE}")
print(f"[설정] POLICY_FEATURE_MODE = {POLICY_FEATURE_MODE}")
print(f"[설정] APPLY_PRICE_CAP_POSTPROCESS = {APPLY_PRICE_CAP_POSTPROCESS}")

#### 함수 정의

In [ ]:
# ============================================================
# 예측 모델 생성 - 셀 3. 공통 함수 (v2 + 정책모듈)
# ============================================================

def qid(col: str) -> str:
    """DuckDB identifier quote"""
    return '"' + str(col).replace('"', '""') + '"'


def qstr(x: str) -> str:
    """DuckDB string quote"""
    return "'" + str(x).replace("'", "''") + "'"


def ensure_date(s):
    return pd.to_datetime(s, errors="coerce").dt.floor("D")


def to_bool_like(x):
    """
    bool / 숫자 / 문자열 True/False/1/0 을 0/1 Series 또는 bool로 변환
    """
    if isinstance(x, pd.Series):
        if x.dtype == bool:
            return x.astype(int)

        if x.dtype == object:
            s = x.astype(str).str.strip().str.lower()
            mapped = s.map({
                "true": 1, "false": 0,
                "1": 1, "0": 0,
                "yes": 1, "no": 0,
                "y": 1, "n": 0,
                "nan": 0, "none": 0, "": 0,
            })
            if mapped.notna().sum() > 0:
                return mapped.fillna(0).astype(int)

        return pd.to_numeric(x, errors="coerce").fillna(0).astype(float).astype(int).clip(lower=0, upper=1)

    return int(bool(x))


def to_week_end_saturday(dt_series: pd.Series) -> pd.Series:
    dt = ensure_date(dt_series)
    return dt + pd.to_timedelta((5 - dt.dt.dayofweek) % 7, unit="D")


def add_policy_flags(date_df: pd.DataFrame) -> pd.DataFrame:
    out = date_df.copy()
    out["date"] = ensure_date(out["date"])
    out["is_policy_excluded"] = False
    out["policy_exclude_label"] = ""

    for r in POLICY_EXCLUDE_RANGES:
        start = pd.to_datetime(r["start"])
        end = pd.to_datetime(r["end"])
        mask = (out["date"] >= start) & (out["date"] <= end)
        out.loc[mask, "is_policy_excluded"] = True
        out.loc[mask, "policy_exclude_label"] = np.where(
            out.loc[mask, "policy_exclude_label"].astype(str).str.len() > 0,
            out.loc[mask, "policy_exclude_label"].astype(str) + "|" + r["label"],
            r["label"],
        )

    final_start = pd.to_datetime(FINAL_TEST_START)
    final_end = pd.to_datetime(FINAL_TEST_END)
    out["is_final_test"] = (out["date"] >= final_start) & (out["date"] <= final_end)

    return out


def load_policy_daily_raw(fuel_key: str) -> Optional[pd.DataFrame]:
    cfg = FUEL_CONFIG[fuel_key]

    if cfg["policy_daily_path"].exists():
        df = pd.read_csv(cfg["policy_daily_path"])
        df["date"] = ensure_date(df["date"])
        return df

    if REQUIRE_STAGE3_POLICY_FILE or (not ALLOW_STEP2_FALLBACK_IF_STAGE3_MISSING):
        raise FileNotFoundError(f"[{fuel_key}] 3단계 정책 일별 파일 없음: {cfg['policy_daily_path']}")

    return None


def load_price_cap_check_weekly(fuel_key: str) -> Optional[pd.DataFrame]:
    cfg = FUEL_CONFIG[fuel_key]
    p = cfg["policy_cap_check_path"]

    if not p.exists():
        return None

    df = pd.read_csv(p)
    if "week_end" in df.columns:
        df["week_end"] = ensure_date(df["week_end"])
    return df


def build_price_cap_proxy_daily(
    fuel_key: str,
    policy_daily_raw: pd.DataFrame,
) -> pd.DataFrame:
    """
    최고가격제 rough retail cap 근사값 생성.
    나중에 교체하기 쉽게 별도 함수로 분리.

    현재 기본 로직:
      retail_margin_proxy_won_l
        = 최근 clean(최고가격제 미적용) 주들의
          [주간 소매평균 - 정유사 실제 세후추정가격] rolling median

      retail_price_cap_approx_won_l
        = 정유사공급가격상한_원L + retail_margin_proxy_won_l
    """
    out = policy_daily_raw[["date"]].copy()
    out["date"] = ensure_date(out["date"])
    out["retail_margin_proxy_won_l"] = np.nan
    out["retail_price_cap_approx_won_l"] = np.nan

    if (not APPLY_PRICE_CAP_POSTPROCESS) or (PRICE_CAP_POSTPROCESS_MODE != "rolling_margin_from_refinery"):
        return out

    if "국내유가_원L" not in policy_daily_raw.columns:
        return out

    weekly = load_price_cap_check_weekly(fuel_key)
    if weekly is None:
        return out

    need_weekly = ["week_end", "정유사_실제세후추정가격_원L", "정유사공급가격상한_원L", "최고가격제_적용여부"]
    missing = [c for c in need_weekly if c not in weekly.columns]
    if missing:
        print(f"[WARN] [{fuel_key}] 최고가격제 점검 파일에 필요한 컬럼 없음: {missing}")
        return out

    daily = policy_daily_raw[["date", "국내유가_원L"]].copy()
    daily["date"] = ensure_date(daily["date"])
    daily["week_end"] = to_week_end_saturday(daily["date"])

    weekly_retail = (
        daily.groupby("week_end", as_index=False)["국내유가_원L"]
        .mean()
        .rename(columns={"국내유가_원L": "retail_weekly_mean_won_l"})
    )

    wk = weekly[need_weekly].copy()
    wk["week_end"] = ensure_date(wk["week_end"])
    wk = wk.rename(columns={
        "정유사_실제세후추정가격_원L": "actual_refinery_posttax_won_l",
        "정유사공급가격상한_원L": "refinery_price_cap_won_l_weekly",
        "최고가격제_적용여부": "price_cap_on_weekly",
    })

    merged = weekly_retail.merge(wk, on="week_end", how="left").sort_values("week_end").reset_index(drop=True)

    merged["retail_weekly_mean_won_l"] = pd.to_numeric(merged["retail_weekly_mean_won_l"], errors="coerce")
    merged["actual_refinery_posttax_won_l"] = pd.to_numeric(merged["actual_refinery_posttax_won_l"], errors="coerce")
    merged["refinery_price_cap_won_l_weekly"] = pd.to_numeric(merged["refinery_price_cap_won_l_weekly"], errors="coerce")
    merged["price_cap_on_weekly"] = to_bool_like(merged["price_cap_on_weekly"])

    merged["retail_margin_raw_won_l"] = (
        merged["retail_weekly_mean_won_l"] - merged["actual_refinery_posttax_won_l"]
    )

    # 최고가격제가 걸리지 않은 주들만 historical margin source로 사용
    clean_margin_source = merged["retail_margin_raw_won_l"].where(merged["price_cap_on_weekly"] == 0)

    rolling_proxy = (
        clean_margin_source.shift(1)
        .rolling(window=PRICE_CAP_MARGIN_LOOKBACK_WEEKS, min_periods=PRICE_CAP_MARGIN_MIN_WEEKS)
        .median()
    )

    expanding_proxy = clean_margin_source.expanding(min_periods=1).median().shift(1)
    global_proxy = clean_margin_source.median()

    merged["retail_margin_proxy_won_l"] = rolling_proxy
    merged["retail_margin_proxy_won_l"] = merged["retail_margin_proxy_won_l"].fillna(expanding_proxy)
    merged["retail_margin_proxy_won_l"] = merged["retail_margin_proxy_won_l"].fillna(global_proxy)

    merged["retail_price_cap_approx_won_l"] = np.where(
        (merged["price_cap_on_weekly"] == 1) & merged["refinery_price_cap_won_l_weekly"].notna(),
        merged["refinery_price_cap_won_l_weekly"] + merged["retail_margin_proxy_won_l"],
        np.nan,
    )

    out_daily = daily[["date", "week_end"]].merge(
        merged[["week_end", "retail_margin_proxy_won_l", "retail_price_cap_approx_won_l"]],
        on="week_end",
        how="left",
    )

    out_daily = out_daily.drop(columns=["week_end"]).drop_duplicates(subset=["date"]).sort_values("date")
    return out_daily


def load_national_table(fuel_key: str) -> pd.DataFrame:
    """
    전국 실제 평균 가격 + 전국 적정가격/하한/상한 + 정책 변수(date-level)

    학습:
      national_actual_price 사용

    예측:
      national_fair_center / national_fair_lower / national_fair_upper 사용
    """
    cfg = FUEL_CONFIG[fuel_key]

    # --------------------------------------------------------
    # 0) 전국 실제가격
    # --------------------------------------------------------
    integrated = pd.read_csv(INTEGRATED_DAILY_PATH)
    integrated["date"] = ensure_date(integrated["date"])

    if cfg["national_actual_col"] not in integrated.columns:
        raise KeyError(f"통합 데이터에 전국 실제가격 컬럼 없음: {cfg['national_actual_col']}")

    nat = integrated[["date", cfg["national_actual_col"]]].copy()
    nat = nat.rename(columns={cfg["national_actual_col"]: "national_actual_price"})
    nat["national_actual_price"] = pd.to_numeric(nat["national_actual_price"], errors="coerce")

    # --------------------------------------------------------
    # 1) stage3 정책 일별 파일 강제 사용
    # --------------------------------------------------------
    fair_source = None
    policy_raw = load_policy_daily_raw(fuel_key)

    if policy_raw is not None:
        if NATIONAL_FAIR_MODE == "policy_applied":
            center_col = "적정가격_정책적용_원L"
            lower_col = "적정범위_정책적용_하한_원L"
            upper_col = "적정범위_정책적용_상한_원L"
            fair_source = "stage3_policy_applied"
        elif NATIONAL_FAIR_MODE == "nonpolicy":
            center_col = "적정가격_미정책_원L"
            lower_col = "적정범위_미정책_하한_원L"
            upper_col = "적정범위_미정책_상한_원L"
            fair_source = "stage3_nonpolicy"
        else:
            raise ValueError(f"NATIONAL_FAIR_MODE는 policy_applied 또는 nonpolicy여야 합니다. 현재: {NATIONAL_FAIR_MODE}")

        need = [center_col, lower_col, upper_col]
        missing = [c for c in need if c not in policy_raw.columns]
        if missing:
            raise KeyError(f"[{fuel_key}] 3단계 정책 파일에 필요한 적정가격 컬럼 없음: {missing}")

        fair = policy_raw[["date", center_col, lower_col, upper_col]].copy()
        fair = fair.rename(columns={
            center_col: "national_fair_center",
            lower_col: "national_fair_lower",
            upper_col: "national_fair_upper",
        })

        # 정책 numeric/text columns
        policy_extra = pd.DataFrame({
            "date": ensure_date(policy_raw["date"]),
            "fuel_tax_cut_pct": pd.to_numeric(policy_raw["유류세인하율_pct"], errors="coerce") if "유류세인하율_pct" in policy_raw.columns else 0.0,
            "fuel_tax_policy_on": to_bool_like(policy_raw["유류세정책_적용여부"]) if "유류세정책_적용여부" in policy_raw.columns else 0,
            "policy_shift_won_l": pd.to_numeric(policy_raw["정책적용_시프트_원L"], errors="coerce") if "정책적용_시프트_원L" in policy_raw.columns else np.nan,
            "price_cap_on": to_bool_like(policy_raw["최고가격제_적용여부"]) if "최고가격제_적용여부" in policy_raw.columns else 0,
            "refinery_price_cap_won_l": pd.to_numeric(policy_raw["정유사공급가격상한_원L"], errors="coerce") if "정유사공급가격상한_원L" in policy_raw.columns else np.nan,
            "price_cap_value_reflected": to_bool_like(policy_raw["최고가격제_수치반영여부"]) if "최고가격제_수치반영여부" in policy_raw.columns else 0,
            "policy_active_any": to_bool_like(policy_raw["정책적용여부_전체"]) if "정책적용여부_전체" in policy_raw.columns else 0,
            "fuel_tax_policy_name": policy_raw["유류세정책명"].astype(str) if "유류세정책명" in policy_raw.columns else np.nan,
            "price_cap_policy_name": policy_raw["최고가격제정책명"].astype(str) if "최고가격제정책명" in policy_raw.columns else np.nan,
            "policy_label": policy_raw["정책라벨"].astype(str) if "정책라벨" in policy_raw.columns else np.nan,
        })

        cap_proxy = build_price_cap_proxy_daily(fuel_key, policy_raw)
        policy_extra = policy_extra.merge(cap_proxy, on="date", how="left")

    # --------------------------------------------------------
    # 2) 예외적으로만 step2 fallback
    # --------------------------------------------------------
    else:
        if not ALLOW_STEP2_FALLBACK_IF_STAGE3_MISSING:
            raise FileNotFoundError(
                f"[{fuel_key}] 3단계 정책 파일이 없고 step2 fallback도 비활성화되어 있습니다.\n"
                f"policy_daily_path={cfg['policy_daily_path']}"
            )

        if not cfg["step2_path"].exists():
            raise FileNotFoundError(f"[{fuel_key}] step2 파일 없음: {cfg['step2_path']}")

        fair = pd.read_csv(cfg["step2_path"])
        fair["date"] = ensure_date(fair["date"])

        need = ["pred_gross", "band_low", "band_high"]
        missing = [c for c in need if c not in fair.columns]
        if missing:
            raise KeyError(f"[{fuel_key}] step2 파일에 필요한 컬럼 없음: {missing}")

        fair = fair[["date", "pred_gross", "band_low", "band_high"]].copy()
        fair = fair.rename(columns={
            "pred_gross": "national_fair_center",
            "band_low": "national_fair_lower",
            "band_high": "national_fair_upper",
        })
        fair_source = "step2_raw_fallback"

        policy_extra = pd.DataFrame({"date": fair["date"]})
        for c in POLICY_NUMERIC_FEATURES:
            policy_extra[c] = np.nan
        for c in POLICY_TEXT_COLUMNS:
            policy_extra[c] = np.nan

    # numeric coercion
    for c in ["national_fair_center", "national_fair_lower", "national_fair_upper"]:
        fair[c] = pd.to_numeric(fair[c], errors="coerce")

    for c in POLICY_NUMERIC_FEATURES:
        if c in policy_extra.columns:
            policy_extra[c] = pd.to_numeric(policy_extra[c], errors="coerce")

    # bool-like numeric cols는 0/1 정리
    for c in ["fuel_tax_policy_on", "price_cap_on", "price_cap_value_reflected", "policy_active_any"]:
        if c in policy_extra.columns:
            policy_extra[c] = pd.to_numeric(policy_extra[c], errors="coerce").fillna(0).astype(int)

    out = nat.merge(fair, on="date", how="left")
    out = out.merge(policy_extra, on="date", how="left")
    out = add_policy_flags(out)

    out["is_train_clean"] = (
        out["national_actual_price"].notna()
        & (~out["is_policy_excluded"])
        & (~out["is_final_test"])
    )

    out["fair_source"] = fair_source

    return out.sort_values("date").reset_index(drop=True)


def inspect_grid_schema() -> pd.DataFrame:
    con = duckdb.connect()
    schema = con.execute(
        f"DESCRIBE SELECT * FROM read_parquet({qstr(str(GRID_PANEL_PATH))})"
    ).df()
    con.close()
    return schema


def select_grid_feature_columns(fuel_key: str) -> List[str]:
    """
    새 컬럼이 계속 추가되는 상황을 가정한 자동 feature selector.

    원칙:
      - 가격 컬럼은 모두 제외
      - target과 같은 날짜 같은 grid의 부분 가격도 leakage라 제외
      - 사후 판정/예측/잔차/적정가격 계열 컬럼 제외
      - 숫자형 컬럼은 자동 포함
    """
    schema = inspect_grid_schema()

    numeric_type_keywords = [
        "INTEGER", "BIGINT", "SMALLINT", "TINYINT",
        "DOUBLE", "FLOAT", "REAL", "DECIMAL", "HUGEINT",
    ]

    numeric_cols = []
    for _, r in schema.iterrows():
        col = r["column_name"]
        typ = str(r["column_type"]).upper()
        if any(k in typ for k in numeric_type_keywords):
            numeric_cols.append(col)

    cfg = FUEL_CONFIG[fuel_key]

    always_exclude = {
        "date",
        "grid_id",
        cfg["grid_price_col"],
    }

    leakage_keywords = [
        "price_mean",
        "actual",
        "pred",
        "fair",
        "적정",
        "하한",
        "상한",
        "deviation",
        "inside",
        "above",
        "below",
        "over",
        "under",
        "residual",
        "target",
        "spread",
        "judge",
        "band",
    ]

    selected = []
    excluded = []

    for col in numeric_cols:
        col_lower = col.lower()

        if col in always_exclude:
            excluded.append((col, "always_exclude"))
            continue

        if any(k.lower() in col_lower for k in leakage_keywords):
            excluded.append((col, "leakage_keyword"))
            continue

        selected.append(col)

    print(f"\n[{cfg['label']}] grid numeric cols: {len(numeric_cols)}")
    print(f"[{cfg['label']}] selected base features: {len(selected)}")
    print(f"[{cfg['label']}] excluded cols: {len(excluded)}")

    return selected


def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    raw year는 넣지 않는다.
    2025~2026 extrapolation에서 tree가 year 값을 이상하게 처리할 수 있기 때문.
    """
    out = df.copy()
    dt = ensure_date(out["date"])

    month = dt.dt.month.astype(float)
    dow = dt.dt.dayofweek.astype(float)
    doy = dt.dt.dayofyear.astype(float)

    out["month_sin"] = np.sin(2 * np.pi * month / 12)
    out["month_cos"] = np.cos(2 * np.pi * month / 12)
    out["dow_sin"] = np.sin(2 * np.pi * dow / 7)
    out["dow_cos"] = np.cos(2 * np.pi * dow / 7)
    out["doy_sin"] = np.sin(2 * np.pi * doy / 366)
    out["doy_cos"] = np.cos(2 * np.pi * doy / 366)

    return out


CALENDAR_FEATURES = [
    "month_sin", "month_cos",
    "dow_sin", "dow_cos",
    "doy_sin", "doy_cos",
]

ANCHOR_FEATURE = "national_anchor_price"


def make_policy_feature_list() -> List[str]:
    if POLICY_FEATURE_MODE == "model_input":
        return list(POLICY_NUMERIC_FEATURES)
    return []


def make_model_feature_list(base_grid_features: List[str]) -> List[str]:
    return list(base_grid_features) + [ANCHOR_FEATURE] + make_policy_feature_list() + CALENDAR_FEATURES


def weighted_mae(y_true, y_pred, weight=None) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    ok = np.isfinite(y_true) & np.isfinite(y_pred)

    if weight is None:
        if ok.sum() == 0:
            return np.nan
        return float(np.mean(np.abs(y_true[ok] - y_pred[ok])))

    w = np.asarray(weight, dtype=float)
    ok = ok & np.isfinite(w) & (w > 0)

    if ok.sum() == 0:
        return np.nan

    return float(np.average(np.abs(y_true[ok] - y_pred[ok]), weights=w[ok]))


def weighted_rmse(y_true, y_pred, weight=None) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    ok = np.isfinite(y_true) & np.isfinite(y_pred)

    if weight is None:
        if ok.sum() == 0:
            return np.nan
        return float(np.sqrt(np.mean((y_true[ok] - y_pred[ok]) ** 2)))

    w = np.asarray(weight, dtype=float)
    ok = ok & np.isfinite(w) & (w > 0)

    if ok.sum() == 0:
        return np.nan

    return float(np.sqrt(np.average((y_true[ok] - y_pred[ok]) ** 2, weights=w[ok])))


def make_train_weight(station_weight: pd.Series) -> np.ndarray:
    w = pd.to_numeric(station_weight, errors="coerce").fillna(0).clip(lower=1)
    return np.sqrt(w).to_numpy(dtype=float)


def recenter_spread_by_date(
    df: pd.DataFrame,
    raw_spread_col: str,
    weight_col: str = "station_weight",
) -> pd.Series:
    """
    날짜별로 predicted spread의 주유소 가중 평균이 0이 되도록 보정.
    """
    tmp = df[["date", raw_spread_col, weight_col]].copy()
    tmp["date"] = ensure_date(tmp["date"])
    tmp[raw_spread_col] = pd.to_numeric(tmp[raw_spread_col], errors="coerce")
    tmp[weight_col] = pd.to_numeric(tmp[weight_col], errors="coerce").fillna(0).clip(lower=0)

    tmp["_num"] = tmp[raw_spread_col] * tmp[weight_col]
    tmp["_den"] = tmp[weight_col]

    g_num = tmp.groupby("date")["_num"].transform("sum")
    g_den = tmp.groupby("date")["_den"].transform("sum")

    weighted_mean = np.where(g_den > 0, g_num / g_den, 0.0)
    return tmp[raw_spread_col] - weighted_mean


def month_ranges(start: str, end: str) -> List[Tuple[pd.Timestamp, pd.Timestamp]]:
    start = pd.to_datetime(start).replace(day=1)
    end = pd.to_datetime(end)

    ranges = []
    cur = start

    while cur <= end:
        nxt = cur + pd.offsets.MonthBegin(1)
        ranges.append((cur, min(nxt, end + pd.Timedelta(days=1))))
        cur = nxt

    return ranges


def apply_policy_postprocess_to_grid_prediction(
    df: pd.DataFrame,
    fuel_key: str,
) -> pd.DataFrame:
    """
    정책 후처리 layer.
    현재는 최고가격제 rough clip만 적용.
    나중에 이 함수만 교체하면 됨.
    """
    out = df.copy()

    out["policy_price_cap_applied"] = 0
    out["policy_price_cap_value_won_l"] = np.nan
    out["policy_price_cap_clip_won_l"] = 0.0

    if (not APPLY_PRICE_CAP_POSTPROCESS) or (PRICE_CAP_POSTPROCESS_MODE != "rolling_margin_from_refinery"):
        return out

    if ("price_cap_on" not in out.columns) or ("retail_price_cap_approx_won_l" not in out.columns):
        return out

    cap_on = pd.to_numeric(out["price_cap_on"], errors="coerce").fillna(0).astype(int) == 1
    cap_value = pd.to_numeric(out["retail_price_cap_approx_won_l"], errors="coerce")

    valid = cap_on & cap_value.notna()
    if valid.sum() == 0:
        return out

    before_center = pd.to_numeric(out["grid_fair_price"], errors="coerce").copy()

    for c in ["grid_fair_price", "grid_fair_lower", "grid_fair_upper"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
        out.loc[valid, c] = np.minimum(out.loc[valid, c], cap_value.loc[valid])

    # 순서 재보정
    lower = np.minimum(out["grid_fair_lower"], out["grid_fair_upper"])
    upper = np.maximum(out["grid_fair_lower"], out["grid_fair_upper"])

    out["grid_fair_lower"] = np.minimum(lower, out["grid_fair_price"])
    out["grid_fair_upper"] = np.maximum(upper, out["grid_fair_price"])

    out.loc[valid, "policy_price_cap_applied"] = 1
    out.loc[valid, "policy_price_cap_value_won_l"] = cap_value.loc[valid]
    out.loc[valid, "policy_price_cap_clip_won_l"] = np.maximum(
        before_center.loc[valid] - out.loc[valid, "grid_fair_price"],
        0.0,
    )

    return out


print("[OK] 공통 함수 정의 완료")

In [ ]:
# ============================================================
# 예측 모델 생성 - 셀 4. 학습 데이터 로드 함수 (v2 + 정책변수 merge)
# ============================================================

def load_training_sample(
    fuel_key: str,
    base_grid_features: List[str],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    clean 기간만 사용해서 학습 샘플 생성.

    target:
      spread_target = grid_price - national_actual_price

    feature:
      grid 구조 변수 + national_actual anchor + (선택적으로 정책 numeric vars) + calendar
    """
    cfg = FUEL_CONFIG[fuel_key]
    label = cfg["label"]

    national = load_national_table(fuel_key)

    national_sql_cols = [
        "date",
        "national_actual_price",
        "is_train_clean",
        "is_policy_excluded",
        "is_final_test",
    ] + POLICY_NUMERIC_FEATURES

    national_for_sql = national[national_sql_cols].copy()
    national_for_sql["date"] = ensure_date(national_for_sql["date"])

    selected_exprs = []
    for c in base_grid_features:
        selected_exprs.append(f"CAST(g.{qid(c)} AS DOUBLE) AS {qid(c)}")

    feature_sql = ",\n            ".join(selected_exprs)

    national_numeric_sql = ",\n            ".join(
        [f"CAST(n.{qid(c)} AS DOUBLE) AS {qid(c)}" for c in POLICY_NUMERIC_FEATURES]
    )

    target_col = cfg["grid_price_col"]
    count_col = cfg["grid_count_col"]

    sql = f"""
        SELECT
            CAST(g.date AS DATE) AS date,
            g.grid_id AS grid_id,

            CAST(g.{qid(target_col)} AS DOUBLE) AS actual_grid_price,
            CAST(g.{qid(count_col)} AS DOUBLE) AS station_weight,
            CAST(n.national_actual_price AS DOUBLE) AS national_actual_price,

            {national_numeric_sql},

            {feature_sql}

        FROM read_parquet({qstr(str(GRID_PANEL_PATH))}) AS g
        INNER JOIN national_dates AS n
            ON CAST(g.date AS DATE) = n.date

        WHERE
            n.is_train_clean = TRUE
            AND g.{qid(target_col)} IS NOT NULL
            AND g.{qid(count_col)} IS NOT NULL
            AND CAST(g.{qid(count_col)} AS DOUBLE) > 0
            AND (hash(g.grid_id) % 1000) < {int(TRAIN_GRID_SAMPLE_PER_MILLE)}
    """

    print("=" * 100)
    print(f"[{label}] 학습 샘플 로드 시작")
    print(f" - target_col: {target_col}")
    print(f" - count_col : {count_col}")
    print(f" - base features: {len(base_grid_features)}")
    print(f" - policy feature mode: {POLICY_FEATURE_MODE}")
    print(f" - grid hash sample: {TRAIN_GRID_SAMPLE_PER_MILLE}/1000")

    con = duckdb.connect()
    con.register("national_dates", national_for_sql)
    df = con.execute(sql).df()
    con.close()

    df["date"] = ensure_date(df["date"])

    df["actual_grid_price"] = pd.to_numeric(df["actual_grid_price"], errors="coerce")
    df["national_actual_price"] = pd.to_numeric(df["national_actual_price"], errors="coerce")
    df["station_weight"] = pd.to_numeric(df["station_weight"], errors="coerce")

    for c in POLICY_NUMERIC_FEATURES:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # target
    df["spread_target"] = df["actual_grid_price"] - df["national_actual_price"]

    # 학습 시 anchor는 전국 실제 평균
    df[ANCHOR_FEATURE] = df["national_actual_price"]

    df = add_calendar_features(df)

    model_features = make_model_feature_list(base_grid_features)

    for c in model_features:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=[
        "date",
        "grid_id",
        "actual_grid_price",
        "national_actual_price",
        "station_weight",
        "spread_target",
    ]).reset_index(drop=True)

    print(f"[{label}] 학습 샘플 row 수: {len(df):,}")
    print(f"[{label}] 학습 날짜 범위: {df['date'].min().date()} ~ {df['date'].max().date()}")
    print(f"[{label}] 학습 grid 수: {df['grid_id'].nunique():,}")
    print(f"[{label}] spread 평균: {df['spread_target'].mean():.4f}")
    print(f"[{label}] spread 표준편차: {df['spread_target'].std():.4f}")

    if POLICY_FEATURE_MODE == "model_input":
        print(f"[WARN] [{label}] 정책기간을 학습에서 제외하므로 정책 feature는 학습 데이터에서 대부분 0/NaN일 수 있습니다.")
        for c in make_policy_feature_list():
            if c in df.columns:
                nn = df[c].notna().sum()
                uniq = df[c].nunique(dropna=True)
                print(f" - {c}: non-null={nn:,}, unique={uniq:,}")

    return df, national


print("[OK] 학습 데이터 로드 함수 정의 완료")

In [ ]:
# ============================================================
# 예측 모델 생성 - 셀 5. 검증 split / LightGBM 학습 함수
# ============================================================

def make_time_folds(
    train_df: pd.DataFrame,
    n_folds: int = N_VALID_FOLDS,
    valid_days: int = VALID_DAYS,
    gap_days: int = GAP_DAYS,
) -> List[Dict]:
    dates = np.array(sorted(pd.to_datetime(train_df["date"].dropna().unique())))

    if len(dates) < valid_days * 2:
        raise ValueError("학습 날짜가 너무 적어서 time validation fold를 만들 수 없습니다.")

    valid_days = min(valid_days, max(60, len(dates) // (n_folds + 2)))

    folds = []

    for i in range(n_folds):
        val_end_idx = len(dates) - (n_folds - 1 - i) * valid_days
        val_start_idx = val_end_idx - valid_days

        train_end_idx = max(0, val_start_idx - gap_days)

        if train_end_idx <= 30:
            continue

        train_dates = dates[:train_end_idx]
        val_dates = dates[val_start_idx:val_end_idx]

        folds.append({
            "fold": i + 1,
            "train_start": pd.to_datetime(train_dates[0]),
            "train_end": pd.to_datetime(train_dates[-1]),
            "valid_start": pd.to_datetime(val_dates[0]),
            "valid_end": pd.to_datetime(val_dates[-1]),
            "train_dates": set(pd.to_datetime(train_dates)),
            "valid_dates": set(pd.to_datetime(val_dates)),
        })

    if len(folds) == 0:
        raise ValueError("생성된 validation fold가 없습니다.")

    return folds


LGBM_PARAM_CANDIDATES = [
    {
        "objective": "huber",
        "learning_rate": 0.045,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 150,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "n_estimators": 1800,
    },
    {
        "objective": "huber",
        "learning_rate": 0.035,
        "num_leaves": 63,
        "max_depth": -1,
        "min_child_samples": 250,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_alpha": 0.2,
        "reg_lambda": 2.0,
        "n_estimators": 2200,
    },
    {
        "objective": "regression_l1",
        "learning_rate": 0.040,
        "num_leaves": 63,
        "max_depth": -1,
        "min_child_samples": 200,
        "subsample": 0.90,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.1,
        "reg_lambda": 1.5,
        "n_estimators": 2000,
    },
    {
        "objective": "huber",
        "learning_rate": 0.030,
        "num_leaves": 127,
        "max_depth": -1,
        "min_child_samples": 350,
        "subsample": 0.80,
        "colsample_bytree": 0.75,
        "reg_alpha": 0.3,
        "reg_lambda": 3.0,
        "n_estimators": 2600,
    },
]


def make_lgbm_model(params: Dict, random_state: int = RANDOM_SEED) -> lgb.LGBMRegressor:
    p = dict(params)
    p.update({
        "random_state": random_state,
        "n_jobs": -1,
        "verbosity": -1,
        "force_col_wise": True,
    })
    return lgb.LGBMRegressor(**p)


def evaluate_baselines(
    train_df: pd.DataFrame,
    folds: List[Dict],
) -> pd.DataFrame:
    rows = []

    for f in folds:
        train_mask = train_df["date"].isin(f["train_dates"])
        valid_mask = train_df["date"].isin(f["valid_dates"])

        tr = train_df.loc[train_mask].copy()
        va = train_df.loc[valid_mask].copy()

        # baseline 0: 모든 grid가 전국 평균과 같다고 보는 경우 => spread = 0
        pred_spread_zero = np.zeros(len(va))
        pred_price_zero = va["national_actual_price"].to_numpy() + pred_spread_zero

        rows.append({
            "model_name": "baseline_spread_zero",
            "fold": f["fold"],
            "valid_start": f["valid_start"],
            "valid_end": f["valid_end"],
            "wmae_spread": weighted_mae(va["spread_target"], pred_spread_zero, va["station_weight"]),
            "wmae_price": weighted_mae(va["actual_grid_price"], pred_price_zero, va["station_weight"]),
            "wrmse_price": weighted_rmse(va["actual_grid_price"], pred_price_zero, va["station_weight"]),
        })

        # baseline 1: grid별 과거 평균 spread
        grid_mean = tr.groupby("grid_id")["spread_target"].mean()
        global_mean = float(tr["spread_target"].mean())

        pred_spread_grid_mean = va["grid_id"].map(grid_mean).fillna(global_mean).to_numpy()
        pred_price_grid_mean = va["national_actual_price"].to_numpy() + pred_spread_grid_mean

        rows.append({
            "model_name": "baseline_grid_mean_spread",
            "fold": f["fold"],
            "valid_start": f["valid_start"],
            "valid_end": f["valid_end"],
            "wmae_spread": weighted_mae(va["spread_target"], pred_spread_grid_mean, va["station_weight"]),
            "wmae_price": weighted_mae(va["actual_grid_price"], pred_price_grid_mean, va["station_weight"]),
            "wrmse_price": weighted_rmse(va["actual_grid_price"], pred_price_grid_mean, va["station_weight"]),
        })

    return pd.DataFrame(rows)


def evaluate_lgbm_params(
    train_df: pd.DataFrame,
    features: List[str],
    folds: List[Dict],
    params: Dict,
    model_name: str,
) -> pd.DataFrame:
    rows = []

    for f in folds:
        train_mask = train_df["date"].isin(f["train_dates"])
        valid_mask = train_df["date"].isin(f["valid_dates"])

        tr = train_df.loc[train_mask].copy()
        va = train_df.loc[valid_mask].copy()

        X_tr = tr[features]
        y_tr = tr["spread_target"]
        w_tr = make_train_weight(tr["station_weight"])

        X_va = va[features]
        y_va = va["spread_target"]
        w_va = make_train_weight(va["station_weight"])

        model = make_lgbm_model(params, random_state=RANDOM_SEED + f["fold"])

        model.fit(
            X_tr,
            y_tr,
            sample_weight=w_tr,
            eval_set=[(X_va, y_va)],
            eval_sample_weight=[w_va],
            eval_metric="l1",
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )

        raw_spread = model.predict(X_va, num_iteration=model.best_iteration_)

        va_pred = va[["date", "station_weight", "national_actual_price", "actual_grid_price", "spread_target"]].copy()
        va_pred["pred_spread_raw"] = raw_spread
        va_pred["pred_spread"] = recenter_spread_by_date(
            va_pred,
            raw_spread_col="pred_spread_raw",
            weight_col="station_weight",
        )
        va_pred["pred_grid_price"] = va_pred["national_actual_price"] + va_pred["pred_spread"]

        rows.append({
            "model_name": model_name,
            "fold": f["fold"],
            "valid_start": f["valid_start"],
            "valid_end": f["valid_end"],
            "best_iteration": int(model.best_iteration_ or params.get("n_estimators", 0)),
            "wmae_spread": weighted_mae(va_pred["spread_target"], va_pred["pred_spread"], va_pred["station_weight"]),
            "wmae_price": weighted_mae(va_pred["actual_grid_price"], va_pred["pred_grid_price"], va_pred["station_weight"]),
            "wrmse_price": weighted_rmse(va_pred["actual_grid_price"], va_pred["pred_grid_price"], va_pred["station_weight"]),
        })

        del model, tr, va, X_tr, X_va
        gc.collect()

    return pd.DataFrame(rows)


def tune_center_model(
    train_df: pd.DataFrame,
    features: List[str],
) -> Tuple[Dict, pd.DataFrame, List[Dict]]:
    folds = make_time_folds(train_df)

    print("\n[검증 fold]")
    for f in folds:
        print(
            f"fold {f['fold']}: "
            f"train {f['train_start'].date()}~{f['train_end'].date()} / "
            f"valid {f['valid_start'].date()}~{f['valid_end'].date()}"
        )

    score_parts = []

    baseline_scores = evaluate_baselines(train_df, folds)
    score_parts.append(baseline_scores)

    for i, params in enumerate(LGBM_PARAM_CANDIDATES, start=1):
        model_name = f"lgbm_center_candidate_{i}"
        print(f"\n[튜닝] {model_name} 평가 시작")
        s = evaluate_lgbm_params(train_df, features, folds, params, model_name)
        s["candidate_idx"] = i
        score_parts.append(s)

        print(
            s.groupby("model_name")[["wmae_price", "wrmse_price"]]
            .mean()
            .round(4)
        )

    scores = pd.concat(score_parts, ignore_index=True)

    summary = (
        scores
        .groupby("model_name", as_index=False)
        .agg(
            mean_wmae_price=("wmae_price", "mean"),
            mean_wrmse_price=("wrmse_price", "mean"),
            mean_wmae_spread=("wmae_spread", "mean"),
            mean_best_iteration=("best_iteration", "mean"),
        )
        .sort_values("mean_wmae_price")
        .reset_index(drop=True)
    )

    print("\n[튜닝 결과 요약]")
    display(summary)

    best_model_name = summary.iloc[0]["model_name"]

    if not str(best_model_name).startswith("lgbm_center_candidate_"):
        raise RuntimeError(
            f"LightGBM이 baseline을 못 이겼습니다. 현재 best={best_model_name}. "
            f"이 경우 feature나 target 구조를 재점검해야 합니다."
        )

    best_idx = int(str(best_model_name).replace("lgbm_center_candidate_", ""))
    best_params = LGBM_PARAM_CANDIDATES[best_idx - 1].copy()

    # best iteration은 fold 평균 기준으로 줄여서 최종 학습 과적합 방지
    best_iter = int(
        summary.loc[summary["model_name"] == best_model_name, "mean_best_iteration"].iloc[0]
    )
    if best_iter > 0:
        best_params["n_estimators"] = max(300, int(best_iter * 1.15))

    return best_params, scores, folds


def fit_final_center_model(
    train_df: pd.DataFrame,
    features: List[str],
    params: Dict,
) -> lgb.LGBMRegressor:
    X = train_df[features]
    y = train_df["spread_target"]
    w = make_train_weight(train_df["station_weight"])

    model = make_lgbm_model(params)
    model.fit(X, y, sample_weight=w)

    return model


print("[OK] 검증 / 모델 학습 함수 정의 완료")

In [ ]:
# ============================================================
# 예측 모델 생성 - 셀 6. 변수 중요도 / feature selection / 잔차 band 모델
# ============================================================

def compute_native_importance(
    model: lgb.LGBMRegressor,
    features: List[str],
) -> pd.DataFrame:
    booster = model.booster_

    gain = booster.feature_importance(importance_type="gain")
    split = booster.feature_importance(importance_type="split")

    imp = pd.DataFrame({
        "feature": features,
        "importance_gain": gain,
        "importance_split": split,
    })

    imp["gain_rank"] = imp["importance_gain"].rank(ascending=False, method="min")
    imp["split_rank"] = imp["importance_split"].rank(ascending=False, method="min")

    return imp.sort_values("importance_gain", ascending=False).reset_index(drop=True)


def fit_one_fold_model_for_importance(
    train_df: pd.DataFrame,
    features: List[str],
    params: Dict,
    fold: Dict,
):
    train_mask = train_df["date"].isin(fold["train_dates"])
    valid_mask = train_df["date"].isin(fold["valid_dates"])

    tr = train_df.loc[train_mask].copy()
    va = train_df.loc[valid_mask].copy()

    model = make_lgbm_model(params, random_state=RANDOM_SEED + 999)

    X_tr = tr[features]
    y_tr = tr["spread_target"]
    w_tr = make_train_weight(tr["station_weight"])

    X_va = va[features]
    y_va = va["spread_target"]
    w_va = make_train_weight(va["station_weight"])

    model.fit(
        X_tr,
        y_tr,
        sample_weight=w_tr,
        eval_set=[(X_va, y_va)],
        eval_sample_weight=[w_va],
        eval_metric="l1",
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )

    return model, va


def compute_permutation_importance_mae(
    model: lgb.LGBMRegressor,
    valid_df: pd.DataFrame,
    features: List[str],
    max_rows: int = 80000,
) -> pd.DataFrame:
    """
    validation set에서 feature 하나를 섞었을 때 MAE가 얼마나 나빠지는지 측정.
    delta_mae가 클수록 중요한 변수.
    """
    va = valid_df.copy()

    if len(va) > max_rows:
        va = va.sample(max_rows, random_state=RANDOM_SEED).reset_index(drop=True)

    X = va[features].copy()
    y = va["spread_target"].to_numpy()
    w = va["station_weight"].to_numpy()

    raw = model.predict(X, num_iteration=model.best_iteration_)
    tmp = va[["date", "station_weight", "spread_target"]].copy()
    tmp["pred_spread_raw"] = raw
    tmp["pred_spread"] = recenter_spread_by_date(tmp, "pred_spread_raw", "station_weight")

    base_mae = weighted_mae(tmp["spread_target"], tmp["pred_spread"], tmp["station_weight"])

    rows = []

    rng = np.random.default_rng(RANDOM_SEED)

    for feat in features:
        Xp = X.copy()
        shuffled = Xp[feat].to_numpy().copy()
        rng.shuffle(shuffled)
        Xp[feat] = shuffled

        raw_p = model.predict(Xp, num_iteration=model.best_iteration_)

        tmp_p = va[["date", "station_weight", "spread_target"]].copy()
        tmp_p["pred_spread_raw"] = raw_p
        tmp_p["pred_spread"] = recenter_spread_by_date(tmp_p, "pred_spread_raw", "station_weight")

        perm_mae = weighted_mae(tmp_p["spread_target"], tmp_p["pred_spread"], tmp_p["station_weight"])

        rows.append({
            "feature": feat,
            "base_mae": base_mae,
            "permuted_mae": perm_mae,
            "delta_mae": perm_mae - base_mae,
        })

    return (
        pd.DataFrame(rows)
        .sort_values("delta_mae", ascending=False)
        .reset_index(drop=True)
    )


def select_features_from_importance(
    importance_df: pd.DataFrame,
    model_features: List[str],
) -> List[str]:
    if not DO_FEATURE_SELECTION:
        return model_features

    imp = importance_df.copy()

    mandatory_keywords = [
        "station_count",
        "gasoline_station_count",
        "diesel_station_count",
        "self_count",
        "brand_count",
        "influence",
        "facility",
        "is_island",
        "is_sea",
        "center_lon",
        "center_lat",
        ANCHOR_FEATURE,
        "month_",
        "dow_",
        "doy_",
    ]

    mandatory = [
        f for f in model_features
        if any(k in f for k in mandatory_keywords)
    ]

    positive_perm = (
        imp.loc[imp["delta_mae"].fillna(0) >= MIN_PERMUTATION_DELTA_MAE, "feature"]
        .tolist()
    )

    top_gain = (
        imp.sort_values("importance_gain", ascending=False)
        .head(MAX_SELECTED_FEATURES)["feature"]
        .tolist()
    )

    selected = list(dict.fromkeys(mandatory + positive_perm + top_gain))
    selected = [f for f in selected if f in model_features]

    if len(selected) == 0:
        selected = model_features

    if len(selected) > MAX_SELECTED_FEATURES:
        # mandatory는 유지하고 나머지는 importance 순서대로 자름
        rest = [f for f in imp["feature"].tolist() if f not in mandatory]
        selected = list(dict.fromkeys(mandatory + rest))[:MAX_SELECTED_FEATURES]

    return selected


def build_oof_residual_frame(
    train_df: pd.DataFrame,
    features: List[str],
    params: Dict,
    folds: List[Dict],
) -> pd.DataFrame:
    """
    잔차 band 모델 학습용 OOF residual 생성.
    in-sample residual을 쓰면 band가 과하게 좁아질 수 있으므로 fold validation residual만 사용.
    """
    parts = []

    for f in folds:
        print(f"[OOF residual] fold {f['fold']}")

        train_mask = train_df["date"].isin(f["train_dates"])
        valid_mask = train_df["date"].isin(f["valid_dates"])

        tr = train_df.loc[train_mask].copy()
        va = train_df.loc[valid_mask].copy()

        model = make_lgbm_model(params, random_state=RANDOM_SEED + 2000 + f["fold"])

        X_tr = tr[features]
        y_tr = tr["spread_target"]
        w_tr = make_train_weight(tr["station_weight"])

        X_va = va[features]
        y_va = va["spread_target"]
        w_va = make_train_weight(va["station_weight"])

        model.fit(
            X_tr,
            y_tr,
            sample_weight=w_tr,
            eval_set=[(X_va, y_va)],
            eval_sample_weight=[w_va],
            eval_metric="l1",
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )

        raw = model.predict(X_va, num_iteration=model.best_iteration_)

        tmp = va[["date", "grid_id", "station_weight", "spread_target"] + features].copy()
        tmp["pred_spread_raw"] = raw
        tmp["pred_spread"] = recenter_spread_by_date(tmp, "pred_spread_raw", "station_weight")
        tmp["local_residual"] = tmp["spread_target"] - tmp["pred_spread"]

        parts.append(tmp)

        del model, tr, va, X_tr, X_va
        gc.collect()

    resid = pd.concat(parts, ignore_index=True)
    return resid


def fit_quantile_residual_models(
    residual_df: pd.DataFrame,
    features: List[str],
) -> Tuple[lgb.LGBMRegressor, lgb.LGBMRegressor, float, float]:
    alpha_low = (1.0 - LOCAL_INTERVAL_COVERAGE) / 2.0
    alpha_high = 1.0 - alpha_low

    base_params = {
        "objective": "quantile",
        "learning_rate": 0.040,
        "num_leaves": 31,
        "min_child_samples": 200,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "n_estimators": 1200,
        "random_state": RANDOM_SEED,
        "n_jobs": -1,
        "verbosity": -1,
        "force_col_wise": True,
    }

    X = residual_df[features]
    y = residual_df["local_residual"]
    w = make_train_weight(residual_df["station_weight"])

    lower_params = dict(base_params)
    lower_params["alpha"] = alpha_low

    upper_params = dict(base_params)
    upper_params["alpha"] = alpha_high

    lower_model = lgb.LGBMRegressor(**lower_params)
    upper_model = lgb.LGBMRegressor(**upper_params)

    lower_model.fit(X, y, sample_weight=w)
    upper_model.fit(X, y, sample_weight=w)

    return lower_model, upper_model, alpha_low, alpha_high


print("[OK] 변수 중요도 / 잔차 band 함수 정의 완료")

In [ ]:
# ============================================================
# 예측 모델 생성 - 셀 7. 2025~2026 예측 / 저장 함수 (v2 + 정책후처리)
# ============================================================

def weighted_group_summary(g: pd.DataFrame) -> pd.Series:
    w = pd.to_numeric(g["station_weight"], errors="coerce").fillna(0).clip(lower=0)
    den = float(w.sum())

    def wavg(col):
        x = pd.to_numeric(g[col], errors="coerce")
        ok = x.notna() & (w > 0)
        if ok.sum() == 0:
            return np.nan
        return float(np.average(x[ok], weights=w[ok]))

    def mean01(col):
        if col not in g.columns:
            return np.nan
        x = pd.to_numeric(g[col], errors="coerce")
        if x.notna().sum() == 0:
            return np.nan
        return float(x.mean())

    return pd.Series({
        "grid_rows": int(len(g)),
        "station_weight_sum": den,

        "actual_grid_price_wavg": wavg("actual_grid_price"),
        "grid_fair_price_wavg": wavg("grid_fair_price"),
        "grid_fair_lower_wavg": wavg("grid_fair_lower"),
        "grid_fair_upper_wavg": wavg("grid_fair_upper"),

        "deviation_wavg": wavg("deviation_from_grid_fair"),

        "inside_rate": float(g["inside_grid_band"].mean()) if g["inside_grid_band"].notna().any() else np.nan,
        "above_rate": float(g["above_grid_band"].mean()) if g["above_grid_band"].notna().any() else np.nan,
        "below_rate": float(g["below_grid_band"].mean()) if g["below_grid_band"].notna().any() else np.nan,

        "fuel_tax_cut_pct_mean": wavg("fuel_tax_cut_pct"),
        "price_cap_on_rate": mean01("price_cap_on"),
        "price_cap_applied_rate": mean01("policy_price_cap_applied"),
        "price_cap_clip_wavg": wavg("policy_price_cap_clip_won_l"),
    })


def load_prediction_chunk(
    fuel_key: str,
    base_grid_features: List[str],
    national: pd.DataFrame,
    start_dt: pd.Timestamp,
    end_dt_exclusive: pd.Timestamp,
) -> pd.DataFrame:
    cfg = FUEL_CONFIG[fuel_key]

    national_pred_cols = [
        "date",
        "national_actual_price",
        "national_fair_center",
        "national_fair_lower",
        "national_fair_upper",
        "is_policy_excluded",
        "is_final_test",
        "policy_exclude_label",
        "fair_source",
    ] + POLICY_NUMERIC_FEATURES + POLICY_TEXT_COLUMNS

    national_pred = national[national_pred_cols].copy()
    national_pred["date"] = ensure_date(national_pred["date"])

    # cell_x, cell_y, center_lon, center_lat은 별도 select 하므로
    # feature_sql에서는 제외해서 duplicate column 방지
    skip_explicit_geo = {"cell_x", "cell_y", "center_lon", "center_lat"}
    feature_cols_sql = [c for c in base_grid_features if c not in skip_explicit_geo]

    selected_exprs = []
    for c in feature_cols_sql:
        selected_exprs.append(f"CAST(g.{qid(c)} AS DOUBLE) AS {qid(c)}")
    feature_sql = ",\n            ".join(selected_exprs) if selected_exprs else "NULL AS _dummy_feature_col"

    national_num_sql = ",\n            ".join(
        [f"CAST(n.{qid(c)} AS DOUBLE) AS {qid(c)}" for c in POLICY_NUMERIC_FEATURES]
    )
    national_txt_sql = ",\n            ".join(
        [f"CAST(n.{qid(c)} AS VARCHAR) AS {qid(c)}" for c in POLICY_TEXT_COLUMNS]
    )

    target_col = cfg["grid_price_col"]
    count_col = cfg["grid_count_col"]

    sql = f"""
        SELECT
            CAST(g.date AS DATE) AS date,
            g.grid_id AS grid_id,
            CAST(g.cell_x AS INTEGER) AS cell_x,
            CAST(g.cell_y AS INTEGER) AS cell_y,
            CAST(g.center_lon AS DOUBLE) AS center_lon,
            CAST(g.center_lat AS DOUBLE) AS center_lat,

            CAST(g.{qid(target_col)} AS DOUBLE) AS actual_grid_price,
            CAST(g.{qid(count_col)} AS DOUBLE) AS station_weight,

            CAST(n.national_actual_price AS DOUBLE) AS national_actual_price,
            CAST(n.national_fair_center AS DOUBLE) AS national_fair_center,
            CAST(n.national_fair_lower AS DOUBLE) AS national_fair_lower,
            CAST(n.national_fair_upper AS DOUBLE) AS national_fair_upper,

            {national_num_sql},
            {national_txt_sql},

            n.is_policy_excluded,
            n.is_final_test,
            n.policy_exclude_label,
            n.fair_source,

            {feature_sql}

        FROM read_parquet({qstr(str(GRID_PANEL_PATH))}) AS g
        INNER JOIN national_pred AS n
            ON CAST(g.date AS DATE) = n.date

        WHERE
            CAST(g.date AS DATE) >= DATE {qstr(str(start_dt.date()))}
            AND CAST(g.date AS DATE) < DATE {qstr(str(end_dt_exclusive.date()))}
            AND g.{qid(count_col)} IS NOT NULL
            AND CAST(g.{qid(count_col)} AS DOUBLE) > 0
    """

    con = duckdb.connect()
    con.register("national_pred", national_pred)
    chunk = con.execute(sql).df()
    con.close()

    if len(chunk) == 0:
        return chunk

    chunk["date"] = ensure_date(chunk["date"])

    for c in POLICY_NUMERIC_FEATURES:
        if c in chunk.columns:
            chunk[c] = pd.to_numeric(chunk[c], errors="coerce")

    # 예측 시 anchor는 전국 적정가격
    chunk[ANCHOR_FEATURE] = pd.to_numeric(chunk["national_fair_center"], errors="coerce")

    chunk = add_calendar_features(chunk)

    if "_dummy_feature_col" in chunk.columns:
        chunk = chunk.drop(columns=["_dummy_feature_col"])

    return chunk


def predict_and_save_final_period(
    fuel_key: str,
    base_grid_features: List[str],
    model_features: List[str],
    center_model: lgb.LGBMRegressor,
    lower_model: lgb.LGBMRegressor,
    upper_model: lgb.LGBMRegressor,
    national: pd.DataFrame,
    output_dir: Path,
) -> Tuple[Path, Path]:
    cfg = FUEL_CONFIG[fuel_key]
    label = cfg["label"]

    pred_path = output_dir / f"{fuel_key}_grid_fair_predictions_2025_2026.parquet"
    summary_path = output_dir / f"{fuel_key}_grid_fair_daily_summary_2025_2026.csv"

    if pred_path.exists():
        pred_path.unlink()

    monthly_ranges = month_ranges(FINAL_TEST_START, FINAL_TEST_END)

    writer = None
    summary_parts = []

    print("=" * 100)
    print(f"[{label}] 2025~2026 예측 저장 시작")
    print(f" - output parquet: {pred_path}")

    for start_dt, end_dt in monthly_ranges:
        chunk = load_prediction_chunk(
            fuel_key=fuel_key,
            base_grid_features=base_grid_features,
            national=national,
            start_dt=start_dt,
            end_dt_exclusive=end_dt,
        )

        if len(chunk) == 0:
            continue

        for c in model_features:
            if c not in chunk.columns:
                chunk[c] = np.nan
            chunk[c] = pd.to_numeric(chunk[c], errors="coerce")

        raw_spread = center_model.predict(chunk[model_features])
        low_resid = lower_model.predict(chunk[model_features])
        high_resid = upper_model.predict(chunk[model_features])

        chunk["predicted_spread_raw"] = raw_spread
        chunk["predicted_spread"] = recenter_spread_by_date(
            chunk,
            raw_spread_col="predicted_spread_raw",
            weight_col="station_weight",
        )

        chunk["predicted_local_residual_low"] = low_resid
        chunk["predicted_local_residual_high"] = high_resid

        chunk["grid_fair_price"] = (
            chunk["national_fair_center"] + chunk["predicted_spread"]
        )

        chunk["grid_fair_lower_raw"] = (
            chunk["national_fair_lower"]
            + chunk["predicted_spread"]
            + chunk["predicted_local_residual_low"]
        )

        chunk["grid_fair_upper_raw"] = (
            chunk["national_fair_upper"]
            + chunk["predicted_spread"]
            + chunk["predicted_local_residual_high"]
        )

        # 안전장치: lower <= center <= upper 보장
        lower = np.minimum(chunk["grid_fair_lower_raw"], chunk["grid_fair_upper_raw"])
        upper = np.maximum(chunk["grid_fair_lower_raw"], chunk["grid_fair_upper_raw"])

        chunk["grid_fair_lower"] = np.minimum(lower, chunk["grid_fair_price"])
        chunk["grid_fair_upper"] = np.maximum(upper, chunk["grid_fair_price"])

        # ----------------------------------------------------
        # 정책 후처리 (최고가격제 rough cap)
        # ----------------------------------------------------
        chunk = apply_policy_postprocess_to_grid_prediction(chunk, fuel_key)

        chunk["deviation_from_grid_fair"] = chunk["actual_grid_price"] - chunk["grid_fair_price"]

        valid_actual = chunk["actual_grid_price"].notna()
        valid_band = chunk[["grid_fair_lower", "grid_fair_upper"]].notna().all(axis=1)

        chunk["inside_grid_band"] = np.where(
            valid_actual & valid_band,
            (chunk["actual_grid_price"] >= chunk["grid_fair_lower"])
            & (chunk["actual_grid_price"] <= chunk["grid_fair_upper"]),
            np.nan,
        )

        chunk["above_grid_band"] = np.where(
            valid_actual & valid_band,
            chunk["actual_grid_price"] > chunk["grid_fair_upper"],
            np.nan,
        )

        chunk["below_grid_band"] = np.where(
            valid_actual & valid_band,
            chunk["actual_grid_price"] < chunk["grid_fair_lower"],
            np.nan,
        )

        chunk["fuel"] = label

        keep_cols = [
            "date",
            "fuel",
            "grid_id",
            "cell_x",
            "cell_y",
            "center_lon",
            "center_lat",
            "station_weight",

            "actual_grid_price",
            "national_actual_price",

            "national_fair_center",
            "national_fair_lower",
            "national_fair_upper",

            # 정책 carry columns
            "fuel_tax_cut_pct",
            "fuel_tax_policy_on",
            "policy_shift_won_l",
            "price_cap_on",
            "refinery_price_cap_won_l",
            "price_cap_value_reflected",
            "policy_active_any",
            "retail_margin_proxy_won_l",
            "retail_price_cap_approx_won_l",
            "fuel_tax_policy_name",
            "price_cap_policy_name",
            "policy_label",

            # 모델 산출
            "predicted_spread_raw",
            "predicted_spread",
            "predicted_local_residual_low",
            "predicted_local_residual_high",

            "grid_fair_price",
            "grid_fair_lower",
            "grid_fair_upper",

            # 정책 후처리 산출
            "policy_price_cap_applied",
            "policy_price_cap_value_won_l",
            "policy_price_cap_clip_won_l",

            "deviation_from_grid_fair",
            "inside_grid_band",
            "above_grid_band",
            "below_grid_band",

            "is_policy_excluded",
            "is_final_test",
            "policy_exclude_label",
            "fair_source",
        ]

        if SAVE_SELECTED_FEATURES_IN_OUTPUT:
            original_selected_features = [
                c for c in model_features
                if (c in base_grid_features) and (c not in keep_cols)
            ]
            keep_cols = keep_cols + original_selected_features

        keep_cols = list(dict.fromkeys(keep_cols))
        out_chunk = chunk[keep_cols].copy()

        daily_summary = (
            out_chunk
            .groupby("date", as_index=False)
            .apply(weighted_group_summary)
            .reset_index(drop=True)
        )
        daily_summary["fuel"] = label
        summary_parts.append(daily_summary)

        table = pa.Table.from_pandas(out_chunk, preserve_index=False)

        if writer is None:
            writer = pq.ParquetWriter(
                pred_path,
                table.schema,
                compression="zstd",
            )

        writer.write_table(table)

        print(f"[{label}] {start_dt.date()} ~ {(end_dt - pd.Timedelta(days=1)).date()} 저장: {len(out_chunk):,} rows")

        del chunk, out_chunk, table
        gc.collect()

    if writer is not None:
        writer.close()
    else:
        raise RuntimeError(f"[{label}] 예측 저장된 row가 없습니다.")

    summary = pd.concat(summary_parts, ignore_index=True)
    summary = summary.sort_values(["date", "fuel"]).reset_index(drop=True)
    summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

    print(f"[{label}] daily summary 저장: {summary_path}")

    return pred_path, summary_path


print("[OK] 예측 / 저장 함수 정의 완료")

In [ ]:
# ============================================================
# 예측 모델 생성 - 셀 8. 유종별 전체 실행 함수 (v2 + 정책메타데이터)
# ============================================================

def run_grid_fair_model_one_fuel(fuel_key: str) -> Dict:
    cfg = FUEL_CONFIG[fuel_key]
    label = cfg["label"]

    output_dir = GRID_FAIR_OUTPUT_DIR / fuel_key
    output_dir.mkdir(parents=True, exist_ok=True)

    model_dir = output_dir / "model"
    model_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 120)
    print(f"[{label}] 그리드 적정가격 모델 시작")
    print("=" * 120)

    # --------------------------------------------------------
    # 1) feature 자동 선택
    # --------------------------------------------------------
    base_grid_features = select_grid_feature_columns(fuel_key)
    model_features_all = make_model_feature_list(base_grid_features)

    # --------------------------------------------------------
    # 2) 학습 데이터 로드
    # --------------------------------------------------------
    train_df, national = load_training_sample(fuel_key, base_grid_features)

    # --------------------------------------------------------
    # 3) 모델 튜닝
    # --------------------------------------------------------
    best_params, tuning_scores, folds = tune_center_model(train_df, model_features_all)

    tuning_scores_path = output_dir / f"{fuel_key}_validation_scores_full_features.csv"
    tuning_scores.to_csv(tuning_scores_path, index=False, encoding="utf-8-sig")
    print(f"[{label}] validation scores 저장: {tuning_scores_path}")

    # --------------------------------------------------------
    # 4) 변수 중요도 산출
    # --------------------------------------------------------
    last_fold = folds[-1]
    imp_model, imp_valid = fit_one_fold_model_for_importance(
        train_df=train_df,
        features=model_features_all,
        params=best_params,
        fold=last_fold,
    )

    native_imp = compute_native_importance(imp_model, model_features_all)
    perm_imp = compute_permutation_importance_mae(
        model=imp_model,
        valid_df=imp_valid,
        features=model_features_all,
        max_rows=80000,
    )

    importance = native_imp.merge(perm_imp, on="feature", how="left")
    importance = importance.sort_values(
        ["delta_mae", "importance_gain"],
        ascending=[False, False],
    ).reset_index(drop=True)

    importance_path = output_dir / f"{fuel_key}_feature_importance.csv"
    importance.to_csv(importance_path, index=False, encoding="utf-8-sig")

    print(f"[{label}] feature importance 저장: {importance_path}")
    display(importance.head(30))

    # --------------------------------------------------------
    # 5) feature selection
    # --------------------------------------------------------
    selected_features = select_features_from_importance(
        importance_df=importance,
        model_features=model_features_all,
    )

    # 정책변수를 모델 입력으로 쓰는 모드면, feature selection에서 빠지지 않게 강제 유지
    if POLICY_FEATURE_MODE == "model_input":
        selected_features = list(dict.fromkeys(selected_features + make_policy_feature_list()))

    print(f"[{label}] 전체 feature 수: {len(model_features_all)}")
    print(f"[{label}] 선택 feature 수: {len(selected_features)}")

    selected_feature_path = output_dir / f"{fuel_key}_selected_features.json"
    with open(selected_feature_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "fuel_key": fuel_key,
                "label": label,
                "base_grid_features": base_grid_features,
                "model_features_all": model_features_all,
                "selected_features": selected_features,
                "policy_feature_mode": POLICY_FEATURE_MODE,
                "policy_numeric_features": POLICY_NUMERIC_FEATURES,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    # --------------------------------------------------------
    # 6) 선택 feature 성능 재검증
    # --------------------------------------------------------
    if DO_FEATURE_SELECTION and set(selected_features) != set(model_features_all):
        print(f"\n[{label}] 선택 feature로 성능 재검증")
        selected_scores = evaluate_lgbm_params(
            train_df=train_df,
            features=selected_features,
            folds=folds,
            params=best_params,
            model_name="lgbm_center_selected_features",
        )

        selected_scores_path = output_dir / f"{fuel_key}_validation_scores_selected_features.csv"
        selected_scores.to_csv(selected_scores_path, index=False, encoding="utf-8-sig")

        full_lgbm_scores = tuning_scores[
            tuning_scores["model_name"].astype(str).str.startswith("lgbm_center_candidate_")
        ].copy()

        full_best_mae = (
            full_lgbm_scores
            .groupby("model_name")["wmae_price"]
            .mean()
            .min()
        )
        selected_mae = selected_scores["wmae_price"].mean()

        print(f"[{label}] full feature best MAE     : {full_best_mae:.4f}")
        print(f"[{label}] selected feature MAE      : {selected_mae:.4f}")

        if selected_mae > full_best_mae * 1.005:
            print(f"[{label}] 선택 feature 성능 저하가 커서 전체 feature 유지")
            final_features = model_features_all
        else:
            print(f"[{label}] 선택 feature 사용")
            final_features = selected_features
    else:
        final_features = model_features_all

    if POLICY_FEATURE_MODE == "model_input":
        final_features = list(dict.fromkeys(final_features + make_policy_feature_list()))

    # --------------------------------------------------------
    # 7) 최종 center model 학습
    # --------------------------------------------------------
    print(f"\n[{label}] 최종 center model 학습")
    center_model = fit_final_center_model(
        train_df=train_df,
        features=final_features,
        params=best_params,
    )

    final_native_imp = compute_native_importance(center_model, final_features)
    final_native_imp.to_csv(
        output_dir / f"{fuel_key}_final_native_importance.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # --------------------------------------------------------
    # 8) OOF residual 생성 후 local interval 모델 학습
    # --------------------------------------------------------
    print(f"\n[{label}] OOF residual 생성")
    residual_df = build_oof_residual_frame(
        train_df=train_df,
        features=final_features,
        params=best_params,
        folds=folds,
    )

    residual_path = output_dir / f"{fuel_key}_oof_residual_sample.parquet"
    residual_df[[
        "date", "grid_id", "station_weight",
        "spread_target", "pred_spread", "local_residual",
    ]].to_parquet(residual_path, index=False, compression="zstd")

    print(f"[{label}] OOF residual 저장: {residual_path}")

    print(f"\n[{label}] local residual quantile model 학습")
    lower_model, upper_model, alpha_low, alpha_high = fit_quantile_residual_models(
        residual_df=residual_df,
        features=final_features,
    )

    # --------------------------------------------------------
    # 9) 모델 저장
    # --------------------------------------------------------
    model_bundle = {
        "fuel_key": fuel_key,
        "label": label,
        "center_model": center_model,
        "lower_residual_model": lower_model,
        "upper_residual_model": upper_model,
        "base_grid_features": base_grid_features,
        "model_features_all": model_features_all,
        "final_features": final_features,
        "best_params": best_params,
        "local_interval_coverage": LOCAL_INTERVAL_COVERAGE,
        "alpha_low": alpha_low,
        "alpha_high": alpha_high,

        "target_definition": "spread_target = actual_grid_price - national_actual_price",
        "prediction_definition": "grid_fair_price = national_fair_center + predicted_spread",
        "recenter_rule": "date-level station-weighted mean(predicted_spread) is forced to zero",

        "national_fair_mode": NATIONAL_FAIR_MODE,
        "require_stage3_policy_file": REQUIRE_STAGE3_POLICY_FILE,
        "policy_feature_mode": POLICY_FEATURE_MODE,
        "policy_numeric_features": POLICY_NUMERIC_FEATURES,
        "policy_text_columns": POLICY_TEXT_COLUMNS,

        "apply_price_cap_postprocess": APPLY_PRICE_CAP_POSTPROCESS,
        "price_cap_postprocess_mode": PRICE_CAP_POSTPROCESS_MODE,
        "price_cap_margin_lookback_weeks": PRICE_CAP_MARGIN_LOOKBACK_WEEKS,
        "price_cap_margin_min_weeks": PRICE_CAP_MARGIN_MIN_WEEKS,
    }

    model_path = model_dir / f"{fuel_key}_grid_fair_model_bundle.joblib"
    joblib.dump(model_bundle, model_path)

    metadata = {
        "fuel_key": fuel_key,
        "label": label,
        "grid_panel_path": str(GRID_PANEL_PATH),
        "integrated_daily_path": str(INTEGRATED_DAILY_PATH),
        "policy_daily_path": str(cfg["policy_daily_path"]),
        "policy_cap_check_path": str(cfg["policy_cap_check_path"]),
        "step2_path": str(cfg["step2_path"]),

        "national_fair_mode": NATIONAL_FAIR_MODE,
        "require_stage3_policy_file": REQUIRE_STAGE3_POLICY_FILE,
        "allow_step2_fallback_if_stage3_missing": ALLOW_STEP2_FALLBACK_IF_STAGE3_MISSING,

        "policy_feature_mode": POLICY_FEATURE_MODE,
        "policy_numeric_features": POLICY_NUMERIC_FEATURES,
        "policy_text_columns": POLICY_TEXT_COLUMNS,

        "apply_price_cap_postprocess": APPLY_PRICE_CAP_POSTPROCESS,
        "price_cap_postprocess_mode": PRICE_CAP_POSTPROCESS_MODE,
        "price_cap_margin_lookback_weeks": PRICE_CAP_MARGIN_LOOKBACK_WEEKS,
        "price_cap_margin_min_weeks": PRICE_CAP_MARGIN_MIN_WEEKS,

        "train_grid_sample_per_mille": TRAIN_GRID_SAMPLE_PER_MILLE,
        "final_test_start": FINAL_TEST_START,
        "final_test_end": FINAL_TEST_END,
        "policy_exclude_ranges": POLICY_EXCLUDE_RANGES,

        "rows_train_sample": int(len(train_df)),
        "n_train_grids": int(train_df["grid_id"].nunique()),
        "train_start": str(train_df["date"].min().date()),
        "train_end": str(train_df["date"].max().date()),

        "base_grid_features": base_grid_features,
        "model_features_all": model_features_all,
        "final_features": final_features,
        "best_params": best_params,
        "local_interval_coverage": LOCAL_INTERVAL_COVERAGE,
        "alpha_low": alpha_low,
        "alpha_high": alpha_high,
    }

    metadata_path = model_dir / f"{fuel_key}_model_metadata.json"
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f"[{label}] 모델 저장: {model_path}")
    print(f"[{label}] 메타데이터 저장: {metadata_path}")

    # --------------------------------------------------------
    # 10) 2025~2026 최종 예측 저장
    # --------------------------------------------------------
    pred_path, summary_path = predict_and_save_final_period(
        fuel_key=fuel_key,
        base_grid_features=base_grid_features,
        model_features=final_features,
        center_model=center_model,
        lower_model=lower_model,
        upper_model=upper_model,
        national=national,
        output_dir=output_dir,
    )

    result = {
        "fuel_key": fuel_key,
        "label": label,
        "output_dir": output_dir,
        "model_path": model_path,
        "metadata_path": metadata_path,
        "importance_path": importance_path,
        "prediction_path": pred_path,
        "summary_path": summary_path,
        "train_rows": len(train_df),
        "train_grids": train_df["grid_id"].nunique(),
        "final_features": final_features,
    }

    print("=" * 120)
    print(f"[{label}] 완료")
    print("=" * 120)

    del train_df, residual_df
    gc.collect()

    return result


print("[OK] 유종별 전체 실행 함수 정의 완료")

#### 예측 실행

In [ ]:
# ============================================================
# 예측 모델 생성 - 셀 9. 휘발유 실행
# ============================================================

gasoline_grid_result = run_grid_fair_model_one_fuel("gasoline")

print("\n[휘발유 결과]")
for k, v in gasoline_grid_result.items():
    if k != "final_features":
        print(f"{k}: {v}")

print("\n[휘발유 final features]")
print(gasoline_grid_result["final_features"])

In [ ]:
# ============================================================
# 예측 모델 생성 - 셀 10. 경유 실행
# ============================================================

diesel_grid_result = run_grid_fair_model_one_fuel("diesel")

print("\n[경유 결과]")
for k, v in diesel_grid_result.items():
    if k != "final_features":
        print(f"{k}: {v}")

print("\n[경유 final features]")
print(diesel_grid_result["final_features"])

#### 시각화

In [ ]:
# ============================================================
# 예측 모델 생성 - 셀 11. 2025~2026 시각화 함수
# ============================================================

def read_prediction_for_date(pred_path: Path, date_value: str) -> pd.DataFrame:
    d = pd.to_datetime(date_value).date()

    con = duckdb.connect()
    df = con.execute(f"""
        SELECT *
        FROM read_parquet({qstr(str(pred_path))})
        WHERE CAST(date AS DATE) = DATE {qstr(str(d))}
    """).df()
    con.close()

    if len(df):
        df["date"] = ensure_date(df["date"])

    return df


def plot_daily_summary_timeseries(
    summary_path: Path,
    fuel_label: str,
    output_dir: Path,
):
    df = pd.read_csv(summary_path)
    df["date"] = ensure_date(df["date"])

    plot_dir = output_dir / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)

    # 1) 실제 평균 vs 그리드 적정가격 평균
    fig, ax = plt.subplots(figsize=(18, 6))
    ax.plot(df["date"], df["actual_grid_price_wavg"], label="실제 그리드 평균가격")
    ax.plot(df["date"], df["grid_fair_price_wavg"], label="그리드 적정가격")
    ax.fill_between(
        df["date"],
        df["grid_fair_lower_wavg"],
        df["grid_fair_upper_wavg"],
        alpha=0.25,
        label="그리드 적정가격대",
    )
    ax.set_title(f"{fuel_label} 2025~2026 그리드 가중평균 실제가격 vs 적정가격대")
    ax.set_xlabel("date")
    ax.set_ylabel("원/L")
    ax.grid(True, alpha=0.3)
    ax.legend()

    out_path = plot_dir / f"{fuel_label}_2025_2026_daily_price_band.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    # 2) deviation
    fig, ax = plt.subplots(figsize=(18, 5))
    ax.plot(df["date"], df["deviation_wavg"], label="실제 - 적정가격")
    ax.axhline(0, linewidth=1)
    ax.set_title(f"{fuel_label} 2025~2026 그리드 가중평균 deviation")
    ax.set_xlabel("date")
    ax.set_ylabel("원/L")
    ax.grid(True, alpha=0.3)
    ax.legend()

    out_path2 = plot_dir / f"{fuel_label}_2025_2026_daily_deviation.png"
    fig.savefig(out_path2, dpi=150, bbox_inches="tight")
    plt.close(fig)

    # 3) inside / above / below rate
    fig, ax = plt.subplots(figsize=(18, 5))
    ax.plot(df["date"], df["inside_rate"], label="범위 내 비율")
    ax.plot(df["date"], df["above_rate"], label="상한 초과 비율")
    ax.plot(df["date"], df["below_rate"], label="하한 미만 비율")
    ax.set_title(f"{fuel_label} 2025~2026 그리드 적정가격대 판정 비율")
    ax.set_xlabel("date")
    ax.set_ylabel("rate")
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.3)
    ax.legend()

    out_path3 = plot_dir / f"{fuel_label}_2025_2026_daily_band_rates.png"
    fig.savefig(out_path3, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"[SAVE] {out_path}")
    print(f"[SAVE] {out_path2}")
    print(f"[SAVE] {out_path3}")


def plot_grid_map_for_date(
    pred_path: Path,
    fuel_label: str,
    date_value: str,
    output_dir: Path,
    value_col: str = "deviation_from_grid_fair",
):
    df = read_prediction_for_date(pred_path, date_value)

    if len(df) == 0:
        print(f"[SKIP] {fuel_label} {date_value}: 예측 row 없음")
        return

    if value_col not in df.columns:
        raise KeyError(f"컬럼 없음: {value_col}")

    plot_dir = output_dir / "plots" / "maps"
    plot_dir.mkdir(parents=True, exist_ok=True)

    d = pd.to_datetime(date_value).date()

    fig, ax = plt.subplots(figsize=(9, 11))

    sc = ax.scatter(
        df["center_lon"],
        df["center_lat"],
        c=df[value_col],
        s=8,
        alpha=0.85,
    )

    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(value_col)

    ax.set_title(f"{fuel_label} {d} grid {value_col}")
    ax.set_xlabel("lon")
    ax.set_ylabel("lat")
    ax.grid(True, alpha=0.2)

    out_path = plot_dir / f"{fuel_label}_{d}_{value_col}.png"
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close(fig)

    print(f"[SAVE] {out_path}")


def make_final_visualizations(result: Dict):
    fuel_label = result["label"]
    output_dir = Path(result["output_dir"])
    pred_path = Path(result["prediction_path"])
    summary_path = Path(result["summary_path"])

    plot_daily_summary_timeseries(
        summary_path=summary_path,
        fuel_label=fuel_label,
        output_dir=output_dir,
    )

    for d in MAP_PLOT_DATES:
        plot_grid_map_for_date(
            pred_path=pred_path,
            fuel_label=fuel_label,
            date_value=d,
            output_dir=output_dir,
            value_col="deviation_from_grid_fair",
        )

        plot_grid_map_for_date(
            pred_path=pred_path,
            fuel_label=fuel_label,
            date_value=d,
            output_dir=output_dir,
            value_col="grid_fair_price",
        )


print("[OK] 시각화 함수 정의 완료")

In [ ]:
make_final_visualizations(gasoline_grid_result)

In [ ]:
make_final_visualizations(diesel_grid_result)